---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>
<h1 align="center">Course: Generative and Agentic AI</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-22: Fine-Tuning LLMs - I</h1>

# Learning Agenda of This Notebook

1. How to Get the Maximum out of a Model?
    - Comparison between Prompt Engineering, RAG and Fine-Tuning
    - What is Fine-Tuning?
    - Key Reasons and Use Cases for Fine-Tuning
2. Pre-Training or Self-Supervised Learning of LLMs
3. Supervised Fine-Tuning (SFT) of Large Language Models
4. Alignment of LLMs (Preference & Reinforcement Learning)
5. Overview of Supervised Fine-Tuning Strategies
6. Hugging Face Developer Ecosystem and Libraries
7. A Deep Dive into the Working of LoRA (Low-Rank Adaptation)
    - LLaMA-7B Model Weights/Parameters
    - The Problem — Why Full Fine-Tuning is Impractical
    - What is Rank of a Matrix?
    - What is Low-Rank Decomposition?
    - How LoRA Exploits The Low-Rank Decomposition?
    - Why Rank Matters for LoRA?
    - Step by Step Working of LoRA
    - Key LoRA Hyperparameters
    - LoRA Architecture Diagram
    - How LoRA Solves the 112 GB Problem
8. Example Questions to Check your Understanding about LoRA

# <span style='background :lightgreen' >1. How to Get the Maximum out of a Model?</span>

## a. Comparison between Prompt Engineering, RAG and Fine-Tuning

| **Aspect**                   | **Prompt Engineering**                                                                | **Fine-Tuning**                                                                 | **Retrieval-Augmented Generation (RAG)**                                    |
| ---------------------------- | ------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------- | --------------------------------------------------------------------------- |
| **Core Idea / Approach**     | Craft effective prompts or few-shot examples to guide the model’s existing knowledge. | Modify the model’s internal weights using new domain-specific data.             | Retrieve relevant external information at query time and add it to prompts. |
| **Data Needs**               | No extra training data; few examples or clear instructions suffice.                   | Hundreds to thousands of high-quality labeled examples for supervised training. | External data sources (documents, PDFs, DBs) stored in a vector database.   |
| **Technical Complexity**     | Low — mostly creative writing and testing.                                            | High — requires ML/engineering skills, GPU setup, and training pipelines.       | Moderate — needs retrieval pipeline, vector DB, and prompt integration.     |
| **Compute & Cost**           | Minimal — inference only.                                                             | High — requires GPU training, time, and expertise.                              | Moderate — retrieval adds latency; no model retraining required.            |
| **Latency / Response Speed** | Fast — no external lookup.                                                            | Fast — inference is quick once model is trained.                                | Slower — retrieval at runtime adds delay (~30–50% overhead).                |
| **Adaptability**             | Very flexible — prompts can be updated instantly.                                     | Static — behavior changes require retraining.                                   | Flexible — update the knowledge base without retraining the model.          |
| **Knowledge Update Method**  | Rephrase or change prompts.                                                           | Retrain or fine-tune the model with new data.                                   | Add or update documents in the vector store.                                |
| **Accuracy & Consistency**   | Variable — depends heavily on prompt quality.                                         | High for structured tasks; risk of overfitting or forgetting unrelated info.    | High — answers are grounded in retrieved sources.                           |
| **Hallucination Risk**       | Higher — model may generate unsupported facts.                                        | Moderate — limited to training data; can still hallucinate outside dataset.     | Lower — answers cite real sources.                                          |
| **Storage Requirements**     | None.                                                                                 | Large — need to store new model checkpoints.                                    | Requires vector embeddings and a database (FAISS, Chroma, etc.).            |
| **Best Use Cases**           | Quick prototyping, creative tasks, UX experimentation.                                | Domain-specific models, style consistency, structured tasks.                    | QA/chatbots needing current or factual information.                         |
| **Maintenance**              | Low — simply edit or discard prompts.                                                 | High — retraining needed periodically to update knowledge or behavior.          | Medium — manage vector DB and retrieval pipeline; no retraining.            |
| **Example**                  | System/role prompt: "You are a helpful assistant…"                                    | Training a legal LLM or medical diagnosis model.                                | Chatbot pulling answers from internal wiki, manuals, or policies.           |



<div style="text-align:center;">
    <img src="../images/ce2.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## b. Comparison between Pre-Training, Fine-Tuning and Alighment

| Aspect | Pre-training | Supervised Fine-Tuning (SFT) | Alignment (Preference Tuning & RL) |
|---|---|---|---|
| **One-line idea** | Learns **WHAT** language and the world look like | Learns **HOW** to answer (follow instructions, chat format) | Learns **WHICH** answers are better (helpful, harmless, honest, correct) |
| **Starting model** | Random weights | Pre-trained base model | SFT model (a frozen copy is kept as the **reference model** `π_ref`) |
| **Training data** | Raw, unlabeled text: web, books, code, math | Curated (prompt, ideal response) pairs written by humans or stronger LLMs | **DPO / RLHF:** (prompt, chosen, rejected) preference pairs<br>**GRPO / RLVR:** prompts only + a reward function (e.g. checks the final math answer) |
| **Data format** | `{"text": "The capital of Pakistan is Islamabad..."}` | `{"prompt": "Capital of Pakistan?", "completion": "Islamabad."}` | `{"prompt": "...", "chosen": "...", "rejected": "..."}` |
| **Typical scale** | Trillions of tokens (Llama 3.1 405B: 15.6T tokens) | Thousands to ~1M examples (InstructGPT: ~13K; Qwen 2: 500K) | Tens of thousands to ~1M comparisons or prompts (InstructGPT: ~33K comparisons + ~31K PPO prompts) |
| **Type of learning** | Self-supervised (the next token is the label) | Supervised (imitate one ground-truth answer) | Preference / reinforcement learning (no single correct answer: a relative signal or a reward) |
| **Step 1: Forward pass** | Predict the next token at **every** position of long text chunks | Prompt + response pass through the model in a chat template | **DPO:** compute log-probs of chosen & rejected under the policy and the frozen reference<br>**RLHF / GRPO:** the model **generates its own answers** (GRPO: a group of 4–64 per prompt), then a reward model or rule-based checker scores them |
| **Step 2: Loss** | Cross-entropy on all tokens:<br>`−log P(next token)` | Cross-entropy on **response tokens only** (prompt tokens masked) | **DPO:** `−log σ(β·[logratio(chosen) − logratio(rejected)])`, logratio = `log π − log π_ref`<br>**PPO:** maximize `reward − β·KL(π ‖ π_ref)`<br>**GRPO:** `A = (r − mean(r)) / std(r)`, loss ≈ `−A·log π + β·KL(π ‖ π_ref)` |
| **Step 3: Backward pass** | Gradients to all weights | All weights (full fine-tuning) or only LoRA adapters | Only the **policy** is updated; reference and reward models stay frozen. PPO also trains a value (critic) model; GRPO needs none |
| **Step 4: Weight update** | AdamW, LR ~1e-4 to 6e-4 (warmup + cosine decay) | AdamW, LR ~1e-5 to 2e-5 (full FT) or ~1e-4 (LoRA); 1–3 epochs | AdamW, LR ~5e-7 to 1e-6; DPO β ≈ 0.1 |
| **Hugging Face TRL trainer** | Not TRL (Megatron-LM, torchtitan, nanoGPT) | `SFTTrainer` | `DPOTrainer`, `RewardTrainer` + `PPOTrainer`, `GRPOTrainer` |

> **Note:** Modern papers call everything after pre-training **post-training**, made of three families: instruction tuning (SFT), preference tuning (RLHF / DPO) and **RL with verifiable rewards (RLVR / GRPO)**. All three stages still use the same four steps (forward → loss → backward → update); what changes is **where the learning signal comes from**: raw text → an ideal answer → a comparison or reward.

## c. Key Reasons and Use Cases for Fine-Tuning
- **Domain Adaptation:** A general-purpose LLM trained on internet text has broad knowledge but lacks deep expertise in specialized fields. Fine-tuning on domain-specific data (medical literature, legal documents, financial reports, scientific papers) gives the model vocabulary, context, and reasoning patterns of that field that were underrepresented in pretraining.
- **Task Specialization:** General LLMs are jack-of-all-trades. Fine-tuning sharpens them for a specific tasks like question answering, code generation, summarization, translation, and classification. (often outperform much larger general models on that specific task)
- **Teaching a Custom Personality or Style:** Fine-tuning allows injecting a specific tone, communication style, or persona into the model. For example, giving an assistant the teaching personality of *Dr. Muhammad Arif Butt* by training the model on his lectures, notes, and explanations — so it responds the way Dr. Arif would.
- **Instruction / Chat Behavior:** Pretrained base models predict next tokens and they do not naturally follow instructions or hold conversations. Fine-tuning on instruction-response pairs teaches the model to behave like a helpful assistant that answers questions, follows commands, and engages in dialogue.
- **Reducing Hallucinations in Specific Domains:** General LLMs hallucinate facts in specialized domains because they lack reliable grounding. Fine-tuning on verified domain data anchors the model to accurate, domain-specific knowledge and reduces confident wrong answers.
- **Working with Private / Proprietary Data:** Organizations cannot send sensitive internal data (patient records, financial data, classified documents) to external APIs like OpenAI. Fine-tuning allows building a private, locally-hosted model trained on internal data with full data privacy.
- **Cost & Latency Efficiency:** A fine-tuned small model (1B–7B) can match or exceed the performance of a large general model (70B+) on a narrow task. Fine-tuning is how you get GPT-4 level performance on your specific task from a model that runs on a laptop.
- **Classification or Threat Detection:** Fine-tuning a pretrained model on phishing emails, spam campaigns, and social engineering messages so it can automatically flag malicious content.
- **Malware Analysis:** Adapting a pretrained model on datasets of malware source code or binary signatures to classify unknown samples as ransomware, trojans, or worms.
- **Vulnerability Identification:** Fine-tuning a pretrained model on code snippets with known security flaws (like SQL injection or buffer overflows) so it can detect vulnerabilities in new applications.
- **Log Anomaly Detection:** Teaching a model using real network or system logs containing attack traces to recognize unusual or malicious activity patterns in real time.
- **Incident Report Summarization:** Fine-tuning a pretrained model on past SOC (Security Operations Center) reports to automatically summarize, triage, or prioritize security incidents.
- **Removing or Adjusting Guardrails:** Pretrained models from major providers are aligned with strict safety guidelines that may block legitimate research, red-team security testing, or culturally specific content. Researchers and security professionals sometimes fine-tune models to adjust or remove these restrictions for controlled, ethical research environments.

# <span style='background :lightgreen' >2. Pre-Training of LLMs (Self-Supervised Learning)</span>

<h2 align="center"><div class="alert alert-success" style="margin: 20px"><b>Pre-Training</b> means teaching a model EVERYTHING about language (NLU + NLG) from scratch, starting from random weights, on massive data (trillions of tokens, i.e., tens of terabytes of text)</div></h2>

> 💡 **Pre-Training** is like a person going through ALL of their education from birth to university:
> - Learning to read → primary school
> - Learning maths → secondary school
> - General knowledge → university
> - Takes years, and is enormously expensive

### Step 1: Forward Pass/Propagation:
- Forward propagation is the process of passing input values through the neural network layer by layer to produce a predicted output. In a classic neural network, each layer applies three operations: **multiplication** (weights), **addition** (biases), and a non-linear **activation function**, then passes the result to the next layer.
- In a transformer-based LLM, every **transformer block** has two main sub-layers:
    - **Self-attention**: each token gathers information from the tokens **before** it (a **causal mask** stops it from looking at future tokens)
    - **MLP (feed-forward network)**: the classic multiply → add → activate operations (e.g., GELU or SwiGLU activation)
    - Plus **residual connections** and **layer normalization** to keep training stable (many modern LLMs, such as Llama, drop the bias terms entirely)
- The final layer produces **logits** (one score per vocabulary token), which **softmax** turns into a probability distribution over the next token. Vocabulary sizes range from ~50K tokens (GPT-2) to ~128K tokens (Llama 3).
- The model predicts the next token at **every position in parallel** in a single forward pass. The targets are simply the same text shifted one position to the left, so one sentence gives many training examples:

```
  Input tokens  →  Embedding  →  Transformer Blocks  →  Output logits  →  Softmax
  ["The", "capital", "of", "Pakistan", "is"]           [seq_len × vocab_size]

  Position   Context the model can see          Target (next token)
  1          "The"                          →   "capital"
  2          "The capital"                  →   "of"
  3          "The capital of"               →   "Pakistan"
  4          "The capital of Pakistan"      →   "is"
  5          "The capital of Pakistan is"   →   "Islamabad"

  At position 5:  P("Islamabad") ≈ 1 / 50,000 = 0.00002   ← very low!
                  (random weights spread probability almost evenly over the whole vocabulary)
```

### Step 2: Loss Calculation:
- The model's predicted output is compared against the actual target tokens to compute a **loss**, a number that quantifies how wrong the prediction was.
- Loss is HIGH when the model assigns LOW probability to the correct next token.
- Loss is ZERO when the model assigns probability 1.0 to the correct token (perfectly confident and correct).
- Common loss functions used in practice:
    - `mse` (Mean Squared Error): Works for regression tasks, i.e., predicting a continuous number (e.g., house price). NOT suitable for next-token prediction because, combined with softmax, it gives very weak gradients when the model is confidently wrong, so learning is slow. Cross-entropy is the natural (maximum-likelihood) loss for probability distributions.
    - `binary_crossentropy`: Works for binary classification (yes/no, spam/not spam). NOT suitable because next-token prediction has tens of thousands of possible classes, not just 2.
    - `categorical_crossentropy`: Used for multi-class classification and for next-token prediction in LLMs, i.e., classifying which of the ~50,000+ vocabulary tokens comes next. Because the labels are token IDs (not one-hot vectors), the exact Keras name is `sparse_categorical_crossentropy`; in PyTorch it is `torch.nn.CrossEntropyLoss` / `F.cross_entropy`, which takes raw logits and applies softmax internally.

```
Loss at one position = −ln( P(correct_token) )        ← natural log
                     = −ln( P("Islamabad") )
                     = −ln(0.00002)
                     = 10.82   ← very high loss (a random model is making a blind guess)

Loss for a batch     = AVERAGE of the per-token losses over ALL positions in ALL sequences
```

> 💡 **Perplexity** = e<sup>Loss</sup> is the loss expressed as "how many tokens is the model choosing between?". At the start, e<sup>10.82</sup> ≈ 50,000, as confused as picking uniformly from the whole vocabulary. A well-trained LLM gets down to a loss of roughly 1.5–2.0 on general text (perplexity ≈ 5–7).

### Step 3: Backward Pass/Propagation:
- Once the loss is computed, we calculate the **derivative of the loss with respect to every weight** in the model. These derivatives are called **gradients**, and they are computed efficiently with the **chain rule** (automatic differentiation, e.g., `loss.backward()` in PyTorch).
- `∂Loss/∂W  →  "if I increase this weight slightly, does the loss go up or down, and by how much?"`
- The gradient flows BACKWARDS, in the reverse order of the forward pass: `Output layer → Block N (MLP → Attention) → … → Block 1 (MLP → Attention) → Embedding layer`
- Every single weight in the model receives a gradient signal. The gradient points in the direction that **increases** the loss, so the update in Step 4 moves each weight in the **opposite** direction.

### Step 4: Weight Update (Gradient Descent + AdamW Optimizer):
- The computed gradients are then used by an **optimization algorithm** to update every weight in the direction that **minimizes the loss**. The simplest version is plain gradient descent:
```
W_new = W_old − learning_rate × gradient
      = W_old − 0.0003 × ∂Loss/∂W
```
- In practice, LLMs use **AdamW**, which replaces the raw gradient with a smoothed, per-weight scaled step:
```
m     = β1 × m + (1 − β1) × g        ← running average of gradients (direction, "momentum")
v     = β2 × v + (1 − β2) × g²       ← running average of squared gradients (magnitude)
W_new = W_old − lr × m̂ / (√v̂ + ε)  −  lr × λ × W_old
                    ↑ Adam step           ↑ decoupled weight decay (the "W" in AdamW)

(g = gradient ∂Loss/∂W,  m̂ and v̂ = bias-corrected m and v)
Typical pre-training values: lr ≈ 1e-4 to 6e-4 (peak), β1 = 0.9, β2 = 0.95, λ = 0.1
```
- After this update: `P("Islamabad") barely moves (e.g., 0.000020 → 0.000021), a tiny improvement`
- Repeat this for TRILLIONS of tokens (Llama 3.1 405B: 15.6T tokens) on thousands of GPUs for weeks to months (millions of GPU-hours), and the weights gradually encode a compressed representation of the language and knowledge in the training data
- This is why it is called **gradient descent**: we descend the loss surface step by step, using the gradient as our compass.
- The **learning rate** is a critical hyperparameter that controls the size of each update step:
    - Too **high** → overshoots the minimum → training diverges or oscillates (loss spikes)
    - Too **low** → takes too long to converge → training is impractically slow
    - Just **right** → steadily descends into a **low-loss region** in a reasonable number of steps. LLMs never find the exact global minimum (and don't need to): with billions of weights there are many equally good low-loss solutions
- In practice the learning rate is not fixed. It follows a **schedule**: a short **warmup** (rising from ~0 to the peak over the first few thousand steps), then a slow **decay** (e.g., cosine) towards a small final value.
- Here is a brief summary of smarter **optimization algorithms** that converge faster and more reliably than basic gradient descent:
<table style="margin-left: 0; margin-right: auto;">
  <thead>
    <tr>
      <th>Optimizer</th>
      <th>Key Idea</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><b>SGD</b> (Stochastic Gradient Descent)</td>
      <td>Updates weights using <b>one random training example</b> at a time instead of the full dataset. It is fast and memory efficient, but updates are very noisy and oscillate around the minimum. (In deep-learning libraries, "SGD", e.g. <code>torch.optim.SGD</code>, is normally run on mini-batches.)</td>
    </tr>
    <tr>
      <td><b>BGD</b> (Batch Gradient Descent)</td>
      <td>Updates weights using the <b>entire dataset</b> for every step. It is very stable and accurate, but very slow, and impossible for LLM-sized datasets</td>
    </tr>
    <tr>
      <td><b>MBGD</b> (Mini-Batch Gradient Descent)</td>
      <td>Updates weights using <b>small batches</b> of data (typically 32–256 examples in classic deep learning). It is a practical compromise between SGD and BGD. LLMs use the same idea at a huge scale: batches of <b>millions of tokens</b> per step (Llama 3 405B: 4M → 16M tokens)</td>
    </tr>
    <tr>
      <td><b>SGD + Momentum</b></td>
      <td>Adds an <b>exponentially weighted running average of past gradients</b> (like a ball rolling downhill that builds up speed) to SGD updates. It smooths out oscillations and accelerates convergence in consistent directions</td>
    </tr>
    <tr>
      <td><b>AdaGrad</b> (Adaptive Gradient)</td>
      <td>Adapts the learning rate <b>per parameter</b> by dividing by the <b>square root</b> of the accumulated sum of past squared gradients. It gives larger updates for rare features and smaller ones for frequent features, but the effective learning rate keeps shrinking and can eventually stop learning</td>
    </tr>
    <tr>
      <td><b>RMSProp</b> (Root Mean Square Propagation)</td>
      <td>Fixes AdaGrad's vanishing learning rate problem by dividing by the square root of an <b>exponentially weighted moving average</b> of squared gradients, so old gradients are gradually forgotten instead of accumulating forever</td>
    </tr>
    <tr>
      <td><b>Adam</b> (Adaptive Moment Estimation)</td>
      <td>Combines <b>SGD + Momentum</b> (tracks the direction of past gradients: 1st moment) and <b>RMSProp</b> (tracks the magnitude of past gradients: 2nd moment), plus a bias correction for the first few steps. Each weight effectively gets its own adaptive step size</td>
    </tr>
    <tr>
      <td><b>AdamW</b> (Adam + Decoupled Weight Decay)</td>
      <td>In Adam, an L2 penalty added to the gradient gets divided by the adaptive term, so weights with large gradients are barely regularized. AdamW <b>decouples weight decay</b> and applies it directly to the weights (<code>W ← W − lr·λ·W</code>), separately from the gradient step. <b>The standard optimizer for pre-training and fine-tuning LLMs</b></td>
    </tr>

  </tbody>
</table>

<br><br>

> 💡 **Self-Supervised Learning:** In LLM pre-training, the four-step cycle above is repeated over **trillions of tokens** from massive text corpora (web pages, books, code, Wikipedia, etc.). No labeling is required, because the **next token itself serves as the label**: labels are AUTOMATICALLY generated FROM THE DATA ITSELF, with no human annotation. (The raw data is, however, heavily filtered, deduplicated and mixed, because data quality matters as much as quantity.) After seeing **trillions of tokens**, the weights gradually encode:
> - Grammar and syntax
> - World knowledge and facts
> - Reasoning patterns
> - Code structure
> - Multilingual understanding
>
> The result is a **base (foundation) model**: it can continue any text, but it does **not** yet follow instructions or behave like an assistant. That is the job of fine-tuning.


### Datasets for Pre-Training
 The following datasets are used in **self-supervised learning** having raw text with no target labels. The model learns by predicting the next token on massive corpora. Use these when you want to adapt a model to a new domain.

| Dataset | Description | Link |
|---|---|---|
| **Common Crawl** | Petabyte-scale web crawl data; the backbone of most LLM pretraining | 🔗 [common-crawl-sample](https://huggingface.co/datasets/agentlans/common-crawl-sample) |
| **C4** | Cleaned version of Common Crawl by Google; used to train T5 | 🔗 [allenai/c4](https://huggingface.co/datasets/allenai/c4) |
| **Wikipedia** | Clean, factual encyclopedic text in 100+ languages | 🔗 [wikimedia/wikipedia](https://huggingface.co/datasets/wikimedia/wikipedia) |
| **Project Gutenberg** | 70,000+ public domain books — great for literary/general language modeling | 🔗 [sedthh/gutenberg_english](https://huggingface.co/datasets/sedthh/gutenberg_english) |
| **ArXiv** | Scientific papers from physics, math, CS, biology etc. | 🔗 [ArXiv Dataset on Kaggle](https://www.kaggle.com/datasets/Cornell-University/arxiv) |
| **GitHub Code** | Large-scale source code across many programming languages | 🔗 [codeparrot/github-code](https://huggingface.co/datasets/codeparrot/github-code) |
| **StackExchange** | Q&A pairs from Stack Overflow and 170+ sister sites | 🔗 [HuggingFaceH4/stack-exchange-preferences](https://huggingface.co/datasets/HuggingFaceH4/stack-exchange-preferences) |
| **The Pile** | 800GB diverse text corpus by EleutherAI combining 22 high-quality sources | 🔗 [EleutherAI/pile](https://huggingface.co/datasets/EleutherAI/pile) |
| **RedPajama** | Open reproduction of LLaMA pretraining data | 🔗 [togethercomputer/RedPajama-Data-1T](https://huggingface.co/datasets/togethercomputer/RedPajama-Data-1T) |


# <span style='background :lightgreen' >3. Supervised Fine-Tuning (SFT) of Large Language Models</span>
<h2 align="center"><div class="alert alert-success" style="margin: 20px"><b>Fine-Tuning</b> means teaching a pretrained/base/foundation model a SPECIFIC skill or behaviour on top of its pre-trained weights, using small, curated, labeled data (megabytes to gigabytes). Its most common form, <b>Supervised Fine-Tuning (SFT)</b> or instruction tuning, turns a base model into an instruction-following assistant</div></h2>


> 💡 **Fine-Tuning** is like a university graduate doing a short specialist course or internship:
> - Learns a specific skill quickly (weeks, not years)
> - Builds on existing knowledge and does not need to start from scratch
> - Much cheaper and faster
> - The core knowledge (language) stays largely intact, as long as the learning rate and number of epochs are kept small
> - Mostly **behaviour** changes (format, style, task skills); it is not an efficient way to add lots of new facts

### Step 1: Forward Pass/Propagation:
- The forward pass works identically to pre-training: input tokens flow layer by layer through embeddings, transformer blocks and output logits.
- The **critical difference** is the starting point. Instead of random weights, fine-tuning begins from pre-trained weights that already encode rich language knowledge. The model is already a competent language user; it just needs to be steered.
- The **input format** is also different. Fine-tuning data is mostly structured as **instruction–response pairs** rather than **raw text**, wrapped in a **chat template**: special tokens that mark who is speaking.
- **Both the prompt and the ideal response are fed into the model together** in one forward pass, so the model predicts every next token in parallel, exactly as in pre-training:

```
Chat template (exact special tokens vary by model):
<|user|> Classify the sentiment of this review: 'The product broke in a week.' <|assistant|> Negative <|eos|>
|-------------------------------- prompt ----------------------------------------|  |---- response ----|

Input tokens  →  Embedding  →  Transformer Blocks  →  Output logits  →  Softmax
                                                       [seq_len × vocab_size]

At the response positions:
   P("Negative" | prompt)              = 0.61   ← already reasonable!
   P(<|eos|>    | prompt + "Negative") = 0.30   ← base model doesn't yet know it should STOP here
   (pre-trained weights know language well, but not the assistant format)
```

> 🔑 **Key Difference:** Pre-training starts from **random weights** (P ≈ 0.00002, close to uniform). Fine-tuning starts from **pre-trained weights** (often P ≈ 0.40–0.80 for the right token), so the model is already in a good region of the loss surface before a single gradient step is taken.

### Step 2: Loss Calculation:
- The loss function is the same: **cross-entropy on next-token predictions**, comparing predicted token probabilities against the target tokens.
- The **critical difference** is *where the loss is computed*. During SFT on instruction data, loss is calculated **only on the response/completion tokens** (including the end-of-turn token), NOT on the instruction/prompt tokens. The model is penalised only for getting the *answer* wrong, not the *question*.
- This is implemented by setting the labels of all prompt positions to `-100`, which PyTorch's cross-entropy ignores (`ignore_index=-100`). It is the default behaviour of Hugging Face TRL's `SFTTrainer` for prompt–completion data.
- Because the model already understands language well, the loss starts **much lower** than in pre-training and converges in far fewer steps.

```
Prompt (masked: labels = -100, no loss computed here):
   "<|user|> Classify the sentiment of this review: 'The product broke in a week.' <|assistant|>"

Completion (loss is computed here, averaged over the response tokens):
Loss = [ −ln P("Negative") − ln P(<|eos|>) ] / 2
     = [ −ln(0.61) − ln(0.30) ] / 2
     = [ 0.49 + 1.20 ] / 2
     = 0.85   ← already low loss (pre-trained weights gave a strong head start)
```
- Training on the `<|eos|>` token is how the model learns **when to stop** answering instead of rambling on.

> 🔑 **Key Difference:** Pre-training computes loss over **every token** in the corpus. SFT masks the prompt and computes loss **only over the response tokens**. It is still next-token prediction, but the model is graded only on its answer, so it learns to *respond* rather than to imitate the user's question.


### Step 3: Backward Pass/Propagation:
- The backward pass is **mechanically identical**: gradients of the loss flow backwards through every transformer block, from the output layer back to the embedding layer.
- The **critical differences** are **where the gradient comes from** and **which weights receive it**:
  - Gradients originate only at the **response tokens** (the masked prompt positions contribute no loss), although they still flow back through the prompt tokens via attention.
  - In **full fine-tuning**, all weights receive gradient updates (same as pre-training).
  - In **parameter-efficient fine-tuning (PEFT)** methods such as **LoRA**, the pre-trained weights are frozen (`requires_grad=False`), so **no weight gradients are computed or stored for them**. The gradient signal still flows backwards *through* the frozen layers (via the activations) to reach the adapters, but only the **small set of adapter parameters** receives weight gradients and gets updated, protecting the general knowledge acquired during pre-training.
  - LoRA adds two small matrices **A** (r × d) and **B** (d × r) beside a frozen weight matrix **W**, so the layer computes `W·x + (α/r)·B·A·x`. **B** starts at zero, so training begins exactly at the pre-trained model. The trainable parameters are often **< 1%** of the model.
- **A note on gradient size:** because the loss is lower, the error signal at the output is usually smaller than in pre-training. However, this is **not** what keeps fine-tuning gentle. AdamW normalizes each update by the running gradient magnitude (`m̂ / √v̂`), so the step size is controlled mainly by the **learning rate** (see Step 4).

```
Output layer → Block N (MLP → Attention) → … → Block 1 → Embedding layer
      ↓
Gradients start only from RESPONSE tokens (prompt positions contribute no loss)
      ↓
Full fine-tuning: every weight W receives ∂Loss/∂W
LoRA:             only the small adapter matrices A and B receive gradients
                  the original weight matrix W stays FROZEN (requires_grad=False)
```

> 🔑 **Key Difference:** Pre-training sends gradients from **every token** to **all weights** to build knowledge from scratch. Fine-tuning sends gradients from **response tokens only**, either to all weights (full fine-tuning) or to a tiny fraction of adapter weights (PEFT/LoRA), to refine behaviour without erasing prior knowledge.


### Step 4: Weight Update (Gradient Descent + AdamW Optimizer):
- The update rule is **the same AdamW formula** as pre-training, but the **hyperparameters are set very differently**:

```
W_new = W_old − learning_rate × AdamW_step(∂Loss/∂W)
      = W_old − 0.000002 × AdamW_step(∂Loss/∂W)   ← full fine-tuning LR is ~10×–100× SMALLER than pre-training

Typical learning rates:
  Pre-training      : ~1e-4 to 6e-4
  Full fine-tuning  : ~2e-6 to 2e-5
  LoRA fine-tuning  : ~1e-4   (higher, because only the small, newly added adapter weights are trained)
```

- After this update: `P("Negative") might go from 0.61 → 0.63 and P(<|eos|>) from 0.30 → 0.33 (gentle refinement)`
- Repeat this for **thousands to millions** of examples, usually for **1–3 epochs**, over **hours to days** on far fewer GPUs than pre-training required.
- A **much smaller learning rate** is essential during full fine-tuning:
    - Too **high** → **catastrophic forgetting**: the model overwrites its pre-trained general knowledge with narrow task-specific behaviour (training for too many epochs on narrow data causes the same problem)
    - Too **low** → the model fails to adapt meaningfully to the new task
    - Just **right** → the model **specialises** on the target task while **retaining** its broad pre-trained capabilities

> 🔑 **Key Difference:** Pre-training uses a **larger learning rate** over **trillions of tokens** to carve knowledge into the weights from scratch. Full fine-tuning uses a **much smaller learning rate** over **thousands to millions of curated examples** (LoRA uses a larger rate, but only on tiny adapter weights) to gently steer behaviour, preserving the pre-trained foundation while building task-specific expertise on top of it.

<br><br>

> 💡 **Supervised Learning:** In fine-tuning, the four-step cycle is repeated over a **small, curated, labeled dataset** of instruction–response pairs. Unlike pre-training, the targets are **not** just the next words of raw text: they are **curated answers**, written by human annotators or, increasingly, **generated by stronger LLMs and then filtered** (synthetic data). Much of the SFT data for models like Llama 3 and Tülu 3 is synthetic. The loss is still next-token prediction; it is called *supervised* because every prompt is paired with a chosen target answer. After seeing **thousands to millions of examples**, the weights are refined to encode:
> - Instruction-following behaviour and the chat format (including when to stop)
> - A desired tone, format or response style
> - Domain-specific terminology and task skills (e.g., medical, legal, coding). SFT is good at teaching the model how to *use* what it already knows, but inefficient at adding many *new facts*; continued pre-training or retrieval (RAG) works better for that
> - Basic safety behaviour (e.g., demonstrations of politely refusing harmful requests); preference-based **alignment** (RLHF/DPO) refines this in the next stage
> - Task-specific reasoning patterns


## Datasets for SFT
- Following datasets are used in **Supervised Fine-Tuning (SFT)** and are formatted as instruction-response pairs or conversations. These teach the model to follow instructions and behave like an assistant.

| Dataset | Description | Link |
|---|---|---|
| **Alpaca Cleaned** | 52K instruction-following examples cleaned from Stanford Alpaca dataset | 🔗 [yahma/alpaca-cleaned](https://huggingface.co/datasets/yahma/alpaca-cleaned) |
| **OpenAssistant (OASST2)** | High-quality human-written multi-turn conversations and ranked responses | 🔗 [OpenAssistant/oasst2](https://huggingface.co/datasets/OpenAssistant/oasst2) |
| **Dolly 15K** | 15K human-generated instruction-response pairs by Databricks employees | 🔗 [databricks/databricks-dolly-15k](https://huggingface.co/datasets/databricks/databricks-dolly-15k) |
| **UltraChat** | Large-scale multi-turn chat dataset with diverse topics | 🔗 [HuggingFaceH4/ultrachat_200k](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k) |
| **ShareGPT** | Real user-ChatGPT conversations shared publicly | 🔗 [anon8231489123/ShareGPT_Vicuna_unfiltered](https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered) |
| **SmolTalk** | High-quality SFT dataset used to train Hugging Face's SmolLM2 models | 🔗 [HuggingFaceTB/smoltalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) |
| **Tulu 3** | State-of-the-art SFT + preference dataset by AllenAI (2024) | 🔗 [allenai/tulu-3-sft-mixture](https://huggingface.co/datasets/allenai/tulu-3-sft-mixture) |
| **arifbutt_dataset** | Personal public information about Dr. Muhammad Arif Butt | 🔗 [arif-butt/arifbutt_dataset](https://huggingface.co/datasets/arif-butt/arifbutt_dataset) |

# <span style='background :lightgreen' >4. Alignment of LLMs (Preference & Reinforcement Learning)</span>

<h2 align="center"><div class="alert alert-success" style="margin: 20px"><b>SFT</b> teaches the model HOW to answer. <b>Alignment</b> teaches the model WHICH answers are better, using preference signals (better vs worse) instead of one correct answer</div></h2>

> 💡 **Alignment** is like a new employee getting feedback from an experienced manager:
> - The employee already knows how to do the job (that was SFT)
> - Nobody writes out the perfect answer; the manager just says *"this draft is better than that one"*
> - Over time, the employee learns the manager's taste: what is helpful, polite, safe and correct
> - Quick and cheap compared with an education, but the feedback must be consistent, or the employee learns the wrong lessons

- Applied **after Supervised Fine-Tuning (SFT)** to improve helpfulness, harmlessness, honesty, tone and reliability (and, in reasoning models, correctness)
- At this stage the model already follows instructions; alignment teaches it **which of its possible responses humans prefer**
- Training uses **preference signals** (chosen vs rejected responses) or **reward scores** instead of direct ground-truth answers
- In modern papers, everything after pre-training is called **post-training**: SFT → preference tuning (RLHF / DPO) → RL with verifiable rewards (RLVR / GRPO)

**Two families of alignment methods:**
```
OFFLINE (no text generation during training)     ONLINE RL (model generates text during training)
─────────────────────────────────────────────    ─────────────────────────────────────────────────
DPO and its variants (IPO, KTO, SimPO, ORPO)     RLHF with PPO, RLAIF, GRPO / RLVR

Fixed dataset of (prompt, chosen, rejected)      Dataset of prompts only
        ↓                                                ↓
Train the policy directly on the pairs           Policy generates answers → a reward model or
                                                 rule-based checker scores them → update → repeat
```

**Common Alignment Techniques:**

- **[RLHF (2022)](https://arxiv.org/abs/2203.02155) Reinforcement Learning from Human Feedback:** Human annotators rank multiple model responses. A **reward model** is trained on these rankings, and the LLM is then optimized with **[PPO](https://arxiv.org/pdf/1707.06347)** to maximize the reward, while a **KL penalty** keeps it close to the SFT model. Popularized by InstructGPT (2022), whose pipeline (SFT → reward model → PPO) was used for early ChatGPT. (The idea of RL from human preferences itself dates back to 2017.)
- **[DPO (2023)](https://arxiv.org/abs/2305.18290) Direct Preference Optimization:** A simpler alternative to RLHF that **directly trains the model on (chosen, rejected) response pairs**, without a separate reward model, without generating text during training, and without a PPO training loop. Mathematically it optimizes the same objective as RLHF (maximize reward with a KL constraint). Easier, more stable, and widely used in modern LLM alignment (e.g., Llama 3, Tülu 3).
- **[RLAIF (2023)](https://arxiv.org/abs/2309.00267) Reinforcement Learning from AI Feedback:** Instead of human annotators, a **strong AI model generates the preference judgments**. This greatly reduces labeling cost and enables large-scale alignment.
- **[GRPO (2024)](https://arxiv.org/pdf/2402.03300) Group Relative Policy Optimization:** Introduced in the DeepSeekMath paper and later used to train reasoning models such as **DeepSeek-R1**. The model generates a **group of candidate answers** per prompt, and each answer's reward is compared with the **group average**, which removes the need for a separate value/critic model. Rewards often come from **rule-based checkers** (is the final answer correct? is the format right?), which is called **RL with Verifiable Rewards (RLVR)**.
- **[Constitutional AI (2022)](https://arxiv.org/abs/2212.08073):** A framework introduced by Anthropic in which the model follows a **set of written principles (a "constitution")**. It has two phases: (1) **supervised:** the model critiques and revises its own answers against the constitution and is fine-tuned on the revisions; (2) **RL:** an AI model compares pairs of responses against the constitution to create preference data, which is used for RL training (RLAIF). This enables scalable alignment with minimal human labeling.

| Technique | Preference / reward comes from | Models in memory | Offline or online |
|---|---|---|---|
| **RLHF (PPO)** | Human rankings → trained reward model | 4 (policy, reference, reward, value/critic) | Online |
| **DPO** | (chosen, rejected) pairs from humans or AI | 2 (policy, reference) | Offline |
| **RLAIF** | AI judge → preference labels → reward model | 4 (when used with PPO) | Online |
| **GRPO** | Rule-based checker or reward model, compared within a group | 2–3 (policy, reference, + reward model if not rule-based) | Online |
| **Constitutional AI** | AI critique & AI preferences guided by a written constitution | Phase 1 like SFT; phase 2 like RLAIF | Both phases |

### Step 1: Forward Pass/Propagation:
- The forward pass works identically to pre-training and SFT: input tokens flow layer by layer through embeddings, transformer blocks and output logits.
- The **critical difference** is that **two models** take part. The **policy** (π) is the SFT model being trained. The **reference model** (π_ref) is a **frozen copy of the SFT model**, used as an anchor so the policy doesn't drift too far.
- The **input** is also different:
    - **DPO:** each example contains **two complete answers** to the same prompt, a *chosen* (better) one and a *rejected* (worse) one. Both are run through **both** models, so one example = 4 forward passes.
    - **RL (PPO / GRPO):** the dataset contains **only prompts**. The policy first **generates its own answers** (sampling, no gradients). These are scored, then run through the policy and reference again to get log-probabilities.
- The output we need is no longer one next-token probability but the **log-probability of a whole answer**: the **sum** of the log-probabilities of its response tokens (prompt tokens masked, as in SFT).

```
Prompt:    "<|user|> What is the capital of Pakistan? <|assistant|>"
Chosen:    "The capital of Pakistan is Islamabad."          ← correct and polite
Rejected:  "Obviously Islamabad, everyone knows that."      ← correct, but rude

DPO forward pass (2 answers × 2 models):
                              policy π (trainable)    reference π_ref (frozen SFT copy)
log P(chosen   | prompt)            −12.4                     −12.4
log P(rejected | prompt)            −11.8                     −11.8
   ← at the start π = π_ref, and the SFT model slightly PREFERS the rude answer (−11.8 > −12.4)

RL (PPO / GRPO) forward pass:
   1. Generate: the policy samples its OWN answers (here a group of 4, no gradients)
        ① "The capital of Pakistan is Islamabad."     ② "Karachi."
        ③ "Islamabad."                                ④ "Obviously Islamabad, duh."
   2. Score:    reward model or rule-based checker gives each answer a reward
   3. Re-run:   policy (with gradients) and reference compute log P(answer | prompt)
```

> 🔑 **Key Difference:** SFT feeds **one ideal answer** written in the dataset. Alignment feeds **two competing answers** (DPO) or **the model's own generated answers** (RL), and always alongside a **frozen reference model**. The question is no longer "what is the next token?" but "how likely is this whole answer?"

### Step 2: Loss Calculation:
- There is no single correct token anymore. The loss measures whether the model **ranks the better answer above the worse one**, or gives higher probability to higher-reward answers.
- **DPO loss** (a binary cross-entropy on the preference):
    - `logratio(y) = log π(y) − log π_ref(y)` measures how much the policy has moved away from the SFT model for answer *y* (DPO calls `β × logratio` the **implicit reward**)
    - `Loss = −ln σ( β × [ logratio(chosen) − logratio(rejected) ] )`, where σ is the sigmoid
    - `β` (typically 0.1) controls how strongly the policy is held close to the reference
- **RLHF (PPO):** a **reward model** is trained first on human rankings with `Loss_RM = −ln σ( r(chosen) − r(rejected) )`. The policy is then trained to **maximize** `reward − β × KL(π ‖ π_ref)`.
- **GRPO:** the rewards of a group of answers are normalized into **advantages** `A = (r − mean(r)) / std(r)`. Answers better than the group average get positive advantage, worse ones negative.

```
DPO
At the start (π = π_ref):
margin = β × [ logratio(chosen) − logratio(rejected) ]
       = 0.1 × [ (−12.4 − (−12.4)) − (−11.8 − (−11.8)) ]
       = 0
Loss   = −ln σ(0) = −ln(0.5) = 0.693        ← DPO loss is ALWAYS 0.693 at step 0

GRPO
rewards     = [1, 0, 1, 0]                  (here: 1 if correct AND polite, else 0)
mean = 0.5, std = 0.5
advantage A = [+1, −1, +1, −1]
Loss        ≈ −mean( A × log π(answer) ) + β × KL(π ‖ π_ref)
              (simplified: the full loss also clips the update size, like PPO)
```

> 🔑 **Key Difference:** The SFT loss is **absolute**: `−ln P(target token)`, i.e., make *this* answer likely. The alignment loss is **relative**: make the chosen answer *more likely than* the rejected one (DPO), or make above-average answers more likely than below-average ones (GRPO). (A known DPO quirk: both log-probabilities can go *down* during training, as long as the rejected one goes down faster.)


### Step 3: Backward Pass/Propagation:
- The backward pass is **mechanically identical**: gradients flow backwards through every transformer block of the **policy**, from the output layer back to the embedding layer.
- The **critical differences** are **which models receive gradients** and **which direction they push**:
  - Only the **policy** is updated (all weights, or LoRA adapters only, as in SFT).
  - The **reference model** is frozen (`torch.no_grad()` / `requires_grad=False`), and so is the **reward model**.
  - **PPO** also trains a separate **value (critic) model** to estimate expected reward; **GRPO** doesn't need one, because the group average serves as the baseline.
  - The **generation step has no gradients**; log-probabilities are recomputed afterwards with gradients enabled.
- The DPO gradient shows what is happening:

```
∇Loss = −β × σ(−margin) × [ ∇log π(chosen) − ∇log π(rejected) ]
              ↑ weight: LARGE when the model ranks the pair WRONGLY, SMALL when it is already right

Policy π:          Output layer → Block N → … → Block 1 → Embedding   ← gradients ✔
Reference π_ref:   frozen                                             ← no gradients ✘
Reward model:      frozen                                             ← no gradients ✘
Value / critic:    PPO only (trained separately)                      ← GRPO doesn't need it

Direction of the push:
   ↑ log π(chosen)     "The capital of Pakistan is Islamabad."
   ↓ log π(rejected)   "Obviously Islamabad, everyone knows that."
```

> 🔑 **Key Difference:** SFT gradients push **one** answer up. Alignment gradients push the better answer **up** and the worse answer **down** at the same time, and only the policy moves: the reference and reward models stay frozen.


### Step 4: Weight Update (Gradient Descent + AdamW Optimizer):
- The update rule is **the same AdamW formula** as pre-training and SFT, but with an **even smaller learning rate**:

```
W_new = W_old − learning_rate × AdamW_step(∂Loss/∂W)
      = W_old − 0.0000005 × AdamW_step(∂Loss/∂W)   ← ~10× SMALLER than SFT

Typical learning rates:
  Pre-training     : ~1e-4 to 6e-4
  SFT (full)       : ~2e-6 to 2e-5
  DPO              : ~5e-7   (β ≈ 0.1)
  PPO / GRPO       : ~1e-6
```

- After some training steps:

```
                        policy π          reference π_ref     logratio
log P(chosen)       −12.4 → −10.4            −12.4              +2.0
log P(rejected)     −11.8 → −12.8            −11.8              −1.0

margin = 0.1 × [ 2.0 − (−1.0) ] = 0.3
Loss   = −ln σ(0.3) = 0.554      (was 0.693)   ← the model now PREFERS the polite answer
```

- Repeat this for **tens of thousands to ~1M** preference pairs (DPO: usually 1–3 epochs over a fixed dataset) or **thousands of generate → score → update iterations** (PPO / GRPO), over **hours to days**.
- The **learning rate** and **β** must be balanced carefully:
    - LR too **high** or β too **low** → **over-optimization**: the policy drifts far from the SFT model and starts **reward hacking**, i.e., producing answers that score well but aren't actually better (e.g., overly long, flattering/sycophantic or over-cautious answers), and fluency can degrade
    - LR too **low** or β too **high** → the model barely changes from the SFT model
    - Just **right** → the model **prefers better answers** while keeping the knowledge and skills from pre-training and SFT (the KL / reference anchor makes sure of that)

> 🔑 **Key Difference:** SFT uses a small learning rate to **imitate** curated answers. Alignment uses an **even smaller** learning rate, plus a **KL anchor to the reference model**, to **re-rank** answers the model can already produce, without forgetting what it learned before.

<br><br>

> 💡 **Preference & Reinforcement Learning:** In alignment, the four-step cycle is repeated over **preference pairs** (DPO) or over the model's **own generated answers** scored by a reward (RLHF / RLAIF / GRPO). Unlike SFT, there is **no single correct answer**: the signal is **relative** ("A is better than B", or "better than the group average"). The preferences come from **human rankers** (RLHF), an **AI judge** (RLAIF, Constitutional AI) or a **rule-based checker** (GRPO / RLVR). After training, the weights are refined to encode:
> - **Helpfulness:** clear, complete, well-structured answers
> - **Harmlessness:** refusing genuinely harmful requests, without over-refusing harmless ones
> - **Honesty:** admitting uncertainty and fewer hallucinations
> - A preferred **tone and style** (polite, concise, not sycophantic)
> - **Reasoning** (with GRPO / RLVR): longer, self-checking chains of thought that reach verifiably correct answers
>
> The result is an **aligned assistant** (or a **reasoning model**). The main risks are **reward hacking**, **sycophancy**, **over-refusal** and the **"alignment tax"** (small drops on some benchmarks).


### Datasets for Alignment Learning
> Used in the **alignment stage** (post-SFT), formatted as **(prompt, chosen, rejected)** response pairs or as scored/ranked responses. These teach the model which responses are more helpful, safe and aligned with human values.

| Dataset | Description | Link |
|---|---|---|
| **Descriptiveness Sentiment (TRL)** | Small sample preference dataset by the Hugging Face TRL team; great for learning DPO/RLHF | 🔗 [trl-internal-testing/descriptiveness-sentiment-trl-style](https://huggingface.co/datasets/trl-internal-testing/descriptiveness-sentiment-trl-style) |
| **UltraFeedback** | ~64K prompts, each with several model responses rated by GPT-4; the *binarized* version uses the best response as *chosen* and a lower-rated one as *rejected*. Widely used for DPO | 🔗 [HuggingFaceH4/ultrafeedback_binarized](https://huggingface.co/datasets/HuggingFaceH4/ultrafeedback_binarized) |
| **Anthropic HH-RLHF** | Human preference pairs (chosen, rejected) for **helpfulness** and **harmlessness**, released by Anthropic | 🔗 [Anthropic/hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf) |
| **OpenAssistant (OASST1)** | Crowd-sourced, human-written and **human-ranked** multi-turn conversations in many languages; the rankings can be turned into preference pairs or used to train reward models | 🔗 [OpenAssistant/oasst1](https://huggingface.co/datasets/OpenAssistant/oasst1) |
| **Orca DPO Pairs** | ~13K DPO pairs built from Orca-style prompts: a stronger model's answer is *chosen*, a weaker model's answer *rejected* | 🔗 [Intel/orca_dpo_pairs](https://huggingface.co/datasets/Intel/orca_dpo_pairs) |
| **Capybara DPO** | ~7K multi-turn preference pairs built with *distilabel* from the Capybara dataset | 🔗 [argilla/distilabel-capybara-dpo-7k-binarized](https://huggingface.co/datasets/argilla/distilabel-capybara-dpo-7k-binarized) |
| **HelpSteer2** | NVIDIA dataset of responses **rated 0–4 on five attributes** (helpfulness, correctness, coherence, complexity, verbosity); mainly used to train **reward models**, and can be converted into preference pairs | 🔗 [nvidia/HelpSteer2](https://huggingface.co/datasets/nvidia/HelpSteer2) |

### Datasets for Chain-of-Thought / GRPO Training
> Used for training models to **think step by step** and solve complex problems. These are increasingly important with the rise of reasoning models like DeepSeek-R1 and OpenAI o1. For **GRPO / RLVR**, each prompt needs a **checkable final answer** (a number, a letter, passing unit tests) so a rule-based reward can score it. Datasets that also contain **full reasoning traces** are often used for **SFT / distillation** before (or instead of) RL.

| Dataset | Description | Link |
|---|---|---|
| **GSM8K** | 8.5K grade-school math word problems requiring multi-step reasoning; short numeric final answers make it easy to verify with a rule-based reward. Also a standard reasoning benchmark | 🔗 [openai/gsm8k](https://huggingface.co/datasets/openai/gsm8k) |
| **MLX GRPO Dataset** | Dataset formatted for GRPO-based reasoning fine-tuning experiments | 🔗 [Goastro/mlx-grpo-dataset](https://huggingface.co/datasets/Goastro/mlx-grpo-dataset) |
| **NuminaMath-CoT** | Large collection of competition-level math problems with step-by-step (chain-of-thought) solutions; used for SFT and as a problem source in open R1 reproductions | 🔗 [AI-MO/NuminaMath-CoT](https://huggingface.co/datasets/AI-MO/NuminaMath-CoT) |
| **OpenR1 Math** | ~220K math problems with reasoning traces generated by DeepSeek-R1; part of Hugging Face's open reproduction of DeepSeek-R1 (Open-R1) | 🔗 [open-r1/OpenR1-Math-220k](https://huggingface.co/datasets/open-r1/OpenR1-Math-220k) |
| **Magpie Reasoning** | ~250K chain-of-thought reasoning traces generated by DeepSeek-R1-Distill-Llama-70B | 🔗 [Magpie-Align/Magpie-Reasoning-V2-250K-CoT-Deepseek-R1-Llama-70B](https://huggingface.co/datasets/Magpie-Align/Magpie-Reasoning-V2-250K-CoT-Deepseek-R1-Llama-70B) |
| **ARC Challenge** | AI2 Reasoning Challenge: multiple-choice science questions requiring multi-step reasoning; the answer letter is easy to verify | 🔗 [allenai/ai2_arc](https://huggingface.co/datasets/allenai/ai2_arc) |

# <span style='background :lightgreen' >5. Overview of Supervised Fine-Tuning Strategies</span>

<h2 align="center"><div class="alert alert-success" style="margin: 20px">Fine-tuning strategies differ in <b>WHICH parameters</b> are updated (all, some, or small new ones) and <b>WHAT data</b> is used (instructions, a specific task, a domain)</div></h2>

> 💡 Think of the pre-trained model as a fully furnished house:
> - **Full fine-tuning** → renovate every room
> - **Partial fine-tuning** → renovate only the top floor, leave the rest untouched
> - **PEFT** → keep the house exactly as it is and add a few small, removable extensions
> - **Instruction fine-tuning** → not *how much* you renovate, but *what for*: turning the house into a hotel that serves guests

- Strategies **(i)–(iii)** answer **"which parameters do we train?"**
- Strategy **(iv)** answers **"what data do we train on?"**
- The two can be **combined**, e.g., instruction fine-tuning *using* LoRA is the most common recipe for open models

### (i) Full Fine-Tuning
- **All model weights** are updated during training
- Retweaking billions of parameters is compute-, memory- and time-expensive
- **Memory:** with AdamW in mixed precision you store, per parameter, the weights, the gradients, an FP32 master copy and two Adam states, roughly **16 bytes/parameter** plus activations:
```
7B model  × 16 bytes  ≈ 112 GB   (+ activations)   → needs several 80 GB GPUs
70B model × 16 bytes  ≈ 1.1 TB   (+ activations)   → needs a multi-node GPU cluster
```
- Impractical on a single GPU for models of 7B+. It needs multi-GPU techniques such as **FSDP** or **DeepSpeed ZeRO**, which split the weights, gradients and optimizer states across GPUs
- Usually gives the **highest task performance**, especially when the new task or domain is very different from pre-training, but at the highest cost
- Highest risk of **catastrophic forgetting**, and every fine-tuned version is a **full copy** of the model (e.g., ~14 GB per 7B checkpoint in BF16)

### (ii) Partial Fine-Tuning (Layer Freezing / Transfer Learning)
- Instead of updating all parameters, most layers are **frozen** (`requires_grad=False`) and only a selected subset of **existing** layers is trained
- A common strategy is to freeze the early layers (which capture general language features) and fine-tune only the **top N transformer blocks** and/or the **LM head** (the final output layer). In classic transfer learning (e.g., BERT), a new **classification head** is added and trained on top of the frozen model
- Much cheaper than full fine-tuning, but it still **overwrites original weights** in the unfrozen layers, and at the same number of trainable parameters it usually **performs worse than PEFT methods** such as LoRA, which spread small changes across *all* layers

### (iii) Parameter-Efficient Fine-Tuning (PEFT)
- In most PEFT methods, **all original model parameters are frozen**; we do not modify the pretrained weights at all
- Instead, a small number of **new trainable parameters** (often **< 1%** of the model) are added to or around the model, and only these are updated during fine-tuning
- Achieves performance **close to full fine-tuning** (often matching it on instruction-following tasks) at a fraction of the compute and memory cost. Because the base weights are untouched, it also **forgets less**
- A single base model can serve **many tasks** by swapping small adapter files (a few MB to a few hundred MB each) instead of storing full model copies
- PEFT methods fall into three families:
    - **Additive:** add new modules or parameters (Adapters, Prefix / Prompt Tuning, IA3)
    - **Reparameterization:** learn a low-rank *update* to existing weights (LoRA, QLoRA, DoRA)
    - **Selective:** unfreeze a tiny subset of *existing* parameters (BitFit). This is the one exception to "original weights are never modified"

**Common PEFT Techniques:**
- **[Adapters (2019)](https://arxiv.org/abs/1902.00751)**: Inserts small **bottleneck feed-forward modules** inside each transformer block, one after the attention sublayer and one after the FFN sublayer. Each adapter projects down to a lower dimension, applies a non-linearity, then projects back up (with a residual connection). Only these adapter weights are trained while the entire base model remains frozen. Drawback: the extra layers run **sequentially**, which adds some **inference latency**.
- **[Prefix Tuning (2021)](https://arxiv.org/abs/2101.00190) / [Prompt Tuning (2021)](https://arxiv.org/abs/2104.08691)**: Learns a short sequence of trainable **"soft prompt" vectors** that are prepended to the input (Prompt Tuning) or to the keys and values of every attention layer (Prefix Tuning). Extremely few parameters, but usually weaker than LoRA, especially on smaller models.
- **[LoRA (2021)](https://arxiv.org/abs/2106.09685)** *(Low-Rank Adaptation)*: Injects small trainable low-rank matrices alongside frozen weight matrices. Instead of updating the full weight matrix **W** (d × d), it learns two small matrices **A** (r × d) and **B** (d × r), with rank r ≪ d (typically 8–64), whose product **ΔW = B×A** approximates the weight update. **B** is initialized to zero, so training starts exactly from the base model. The original paper adapted only the attention projections (W<sub>q</sub>, W<sub>v</sub>); today LoRA is usually applied to **all linear layers** (attention + MLP). After training, **ΔW can be merged into W**, so there is **no extra inference latency**.
- **[BitFit (2022)](https://arxiv.org/abs/2106.10199)**: Trains **only the bias terms** of the existing (otherwise frozen) layers. Despite updating less than 0.1% of the parameters, it achieves surprisingly competitive results on many NLU benchmarks with BERT-style models. It is one of the most lightweight PEFT techniques, but it **modifies existing parameters** (selective PEFT), and it **cannot be used** with many modern LLMs (e.g., Llama), whose linear layers have **no bias terms**.
- **[IA3 (2022)](https://arxiv.org/abs/2205.05638)** *(Infused Adapter by Inhibiting and Amplifying Inner Activations)*: Introduces three small learned **scaling vectors** that rescale the keys, values and intermediate feed-forward activations inside each transformer layer. It is even more parameter-efficient than LoRA (around 0.01% of the parameters), because it learns only vectors of scalars that **amplify or suppress** existing activations, rather than full matrices.
- **[QLoRA (2023)](https://arxiv.org/abs/2305.14314)** *(Quantized LoRA)*: A memory-efficient variant of LoRA that quantizes the frozen base model weights to **4-bit NF4** precision (plus *double quantization* and *paged optimizers* to save more memory). It then trains standard LoRA adapters on top in **16-bit (BF16)** precision. This makes it possible to fine-tune a 7B model on a single consumer GPU, and the paper fine-tuned a **65B model on a single 48 GB GPU**. Trade-off: training is slower, because the 4-bit weights have to be de-quantized on the fly.
- **[DoRA (2024)](https://arxiv.org/abs/2402.09353)** *(Weight-Decomposed Low-Rank Adaptation)*: An improved variant of LoRA that decomposes each pre-trained weight matrix into a **magnitude** (how large) and a **direction** (which way) component. It trains the magnitude directly (a small vector) and applies LoRA to the *direction component*. This makes its learning behaviour closer to full fine-tuning and often performs better than standard LoRA, especially at low ranks.

### (iv) Instruction Fine-Tuning
- The most common form of SFT. It teaches a **base model** to follow instructions and behave like a helpful assistant
- The pretrained base model knows language but only **continues text**; it does not naturally respond to instructions. Instruction fine-tuning bridges this gap (popularized by **FLAN** and **InstructGPT**)
- It is defined by the **data**, not by the parameters, so it can be done with **full fine-tuning, partial fine-tuning or PEFT** (most often LoRA/QLoRA)
- Each example is formatted with the model's **chat template**, and the loss is computed **only on the response tokens** (see Section 3)
- Training data includes:
    - Task demonstrations (e.g., "Summarise this…", "Translate this…", "Classify this…")
    - Question–answer pairs
    - Multi-turn conversations
    - Safety demonstrations (e.g., politely refusing harmful requests)
- The data is written by human annotators or, increasingly, **generated by stronger LLMs and then filtered** (synthetic data)
- Related, but different in goal:
    - **Task-specific fine-tuning:** trains on a single task (e.g., only sentiment classification) → a specialist, not a general assistant
    - **Domain adaptation / continued pre-training:** trains on raw domain text (e.g., medical papers) with the pre-training loss → adds domain *knowledge* rather than instruction-following *behaviour*

### Summary: Comparison of Fine-Tuning Strategies

| Strategy | Parameters trained | GPU memory (7B model, approx.) | Performance | Forgetting risk | Inference overhead |
|---|---|---|---|---|---|
| **Full fine-tuning** | 100% | ~112 GB + activations (multi-GPU) | Highest | High | None |
| **Partial fine-tuning** | Selected top layers / head | Medium | Medium | Medium (in unfrozen layers) | None |
| **Adapters** | ~1–5% (new modules) | Low | Close to full | Low | Small extra latency |
| **Prefix / Prompt Tuning** | < 0.1% | Low | Lower, especially on small models | Very low | Slightly longer sequences |
| **LoRA** | ~0.1–1% | ~16–20 GB (BF16 base) | Close to full | Low | None (after merging) |
| **QLoRA** | ~0.1–1% | ~6–10 GB (4-bit base) | Close to LoRA | Low | None (after merging) |
| **DoRA** | ~0.1–1% | Slightly above LoRA | Often better than LoRA | Low | None (after merging) |
| **BitFit** | < 0.1% (biases only) | Low | Good on small NLU tasks | Low | None |
| **IA3** | ~0.01% | Low | Good on few-shot tasks | Very low | Negligible |

> 🔑 **Key takeaway:** For most practical LLM fine-tuning today, the default is **instruction fine-tuning with LoRA** (or **QLoRA** on limited hardware). Full fine-tuning is reserved for cases where you have the compute and need the last bit of performance, or a large shift in domain.

# <span style='background :lightgreen' >5. Overview of Supervised Fine-Tuning Strategies of LLMs</span>
### (i) Full Fine-Tuning
- All model weights and parameters are updated during training
- Retweaking billions of parameters is compute/memory expensive and time consuming
- Requires the same order of memory as pretraining — typically impractical for models 7B+
- Produces the best possible task performance but at the highest cost
### (ii) Partial Fine-Tuning (Layer Freezing / Transfer Learning)
- Instead of updating all parameters, most layers are **frozen** (weights locked) and only a selected subset of layers is trained
- Common strategy is to freeze the early layers (which capture general language knowledge) and fine-tune only the **top N transformer blocks** or the **LM Head** (final classification/generation layers)
- Much cheaper than full fine-tuning but less flexible than PEFT
### (iii) Parameter Efficient Fine-Tuning (PEFT)
- In PEFT, **all original model parameters are frozen** — we do not modify the pretrained weights at all
- Instead, a small number of **new trainable parameters** are added to or around the model, and only these are updated during fine-tuning
- Achieves performance close to full fine-tuning at a fraction of the compute and memory cost

**Common PEFT Techniques:**
- **[Adapters (2019)](https://arxiv.org/abs/1902.00751)**: Inserts small **bottleneck feed-forward modules** inside each transformer block — one after the attention sublayer and one after the FFN sublayer. Each adapter projects down to a lower dimension, applies a non-linearity, then projects back up. Only these adapter weights are trained while the entire base model remains frozen.
- **[LoRA (2021)](https://arxiv.org/abs/2106.09685)** *(Low-Rank Adaptation)*: Injects small trainable low-rank matrices alongside the frozen weight matrices of attention layers. Instead of updating the full weight matrix W, it learns two small matrices A and B whose product **ΔW = B×A** approximates the weight update — keeping the base model completely frozen
- **[BitFit (2022)](https://arxiv.org/abs/2106.10199)**:  Trains **only the bias terms** of the existing frozen weight matrices. Despite updating less than 0.1% of total parameters it achieves surprisingly competitive results on many NLP benchmarks — making it the most lightweight PEFT technique of all
- **[IA3 (2022)](https://arxiv.org/abs/2205.05638)** *(Infused Adapter by Inhibiting and Amplifying Inner Activations)*: Introduces three small learned **scaling vectors** that rescale the keys, values and feed-forward activations inside each transformer layer.  It is more parameter-efficient than LoRA as it adds virtually no new parameters and use the learned scalars that amplify or suppress existing activations
- **[QLoRA (2023)](https://arxiv.org/abs/2305.14314)** *(Quantized LoRA)*:  Memory-efficient variant of LoRA that quantizes the frozen base model weights to **4-bit NF4 precision** to dramatically reduce memory usage. It then applies standard LoRA trainable adapters on top in full precision, making it possible to fine-tune 7B+ parameter models on a single consumer GPU
- **[DoRA (2024)](https://arxiv.org/abs/2402.09353)** *(Weight-Decomposed LoRA)*: Improved variant of LoRA that decomposes each pre-trained weight matrix into **magnitude** (how large) and **direction** (which way) components. It then applies LoRA only to the *direction component*, resulting in more stable training and better performance than standard LoRA especially at low ranks
    
    
### (iv) Instruction Fine-Tuning
- A specialized form of SFT that teaches the model to follow instructions and behave like a helpful assistant
- The pretrained base model knows language but does not naturally respond to instructions;  instruction fine-tuning bridges this gap
- Training data includes:
    - Task demonstrations
    - Question-Answer pairs
    - Multi-turn conversations

# <span style='background :lightgreen' >6. Hugging Face Developer Ecosystem and Libraries</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The Hugging Face ecosystem consists of multiple libraries that work together to make training, fine-tuning, and deploying AI models easier.</div></h3>

<pre>
┌─────────────────────────────────────────────────────────────────┐
│                        <b>unsloth</b>                                  │
│   2-5× faster kernels, 60% less VRAM, GGUF export               │
│      (Accelerates training built on Transformers + PEFT + TRL)  │
├────────────────────────────┬────────────────────────────────────┤
│          <b>peft</b>              │          <b>trl</b>                       │
│  LoRA/QLoRA/DoRA           │  Alignment & RLHF training         │
│  Freeze base model         │  SFT, DPO, PPO, GRPO, Reward models│
│  Train only adapters       │  Built on Transformers + PEFT      │
├────────────────────────────┴────────────────────────────────────┤
│                    <b>transformers</b>                                 │
│ Hub Integration, Models, Tokenizers, Configs, Pipelines, Trainer│
│         updates ALL model weights during fine-tuning            │
├─────────────────────────────────────────────────────────────────┤
│              <b>PyTorch / TensorFlow / Keras / JAX</b>                 │
│    (Core DL libraries on which above libraries are built)       │
└─────────────────────────────────────────────────────────────────┘
</pre>



>- **PyTorch** is a flexible, pythonic deep learning framework widely used for research and LLM development.
>- **TensorFlow** is a production-oriented ML/DL framework with strong deployment ecosystem.
>- **Keras** is a high-level user-friendly API for building models, now integrated with TensorFlow.
>- **JAX** (Just After eXecution) is a ML framework that provides fast array computing with automatic differentiation and XLA compilation.

## a. **[`transformers` (2018)](https://arxiv.org/pdf/1910.03771)**: Core Library

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The single most important library to understand is <b>Transformers</b>, every other library in this ecosystem (peft, trl, unsloth) is built on top of it.</div></h3>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Hugging Face Transformers is a library that provides tools for <b>using, loading, fine-tuning,</b> and <b>pre-training</b> open-source AI models with a unified API that works seamlessly with Meta's PyTorch and Google's TensorFlow/Keras and JAX.</div></h3>

- **What it is:** The core Hugging Face library that provides pre-built model architectures, tokenizers, and training utilities for 200+ model families (Llama, Qwen, Mistral, Phi, GPT-2 etc.)
- **What it does:**
    - Load any pretrained model from Hugging Face Hub in one line
    - Tokenize text into model-ready inputs
    - Run inference and training via `Trainer` class
    - Handle model configurations, saving, and pushing to Hub
- **Example(s):**
```python
# HF Hub
from huggingface_hub import login, HfApi
login(token=hf_token)                        
client = HfApi()                             
models_gen = client.list_models(author="arif-butt") 
# Pipeline
from transformers import pipeline
pipe = pipeline("text-generation")
print(pipe("Which is the highest mountain peak in Pakistan?"))
# Models
from transformers import AutoModelForCausalLM
model     = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
# Tokenizers
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
# Config
from transformers import AutoConfig
config = AutoConfig.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
# Trainer
from transformers import Trainer, TrainingArguments
training_args = TrainingArguments(...)
trainer = Trainer(
                model         = model,
                args          = training_args,
                train_dataset = train_dataset,
                eval_dataset  = eval_dataset,
                )
trainer.train()
```
- 🔗 [Documentation](https://huggingface.co/docs/transformers) | 📄 [Paper](https://arxiv.org/abs/1910.03771)


## b. **[`peft` (2021)](https://huggingface.co/docs/peft/index)** (Parameter Efficient Fine-Tuning)
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px"><b>peft</b> is a HF library used to fine-tune transformer-based models by injecting small trainable adapter modules while keeping the massive base model weights completely <b>frozen</b>. It has no training loop of its own — the adapters it injects are trained by <code>transformers</code>' <code>Trainer</code> or by <code>trl</code>'s trainers, so they can be used for <b>SFT</b>, <b>continued pre-training</b> and <b>alignment</b> (DPO, GRPO etc.), but not for pre-training a model from scratch.</div></h3>

- **Problem it solves:** `transformers`'s built-in `Trainer` updates ALL model weights during fine-tuning — requiring enormous GPU memory (112 GB for 7B model). `peft` reduces this to a tiny fraction by only training small adapter matrices
- **What it is:** Hugging Face library that implements all PEFT techniques — LoRA, QLoRA, Adapters, IA³, DoRA etc.
- **What it does:**
    - Wraps any `transformers` model with LoRA adapter layers in a few lines
    - Freezes base model weights automatically — only adapter weights are trained
    - Saves only the small adapter weights (MBs) instead of the full model (GBs)
    - Supports merging adapters back into base model via `merge_and_unload()`
- **When to use:** Whenever you want to fine-tune efficiently without updating all model weights — i.e., almost always. 
- **Sits on top of:** `transformers` — cannot work without it
- **Important Links:**
    - 🔗 [Documentation](https://huggingface.co/docs/peft)
    - 📄 [LoRA Paper](https://arxiv.org/abs/2106.09685)
    - [GitHub](https://github.com/huggingface/peft)
- **Example PEFT Pipeline (Manual Adapter Injection):**
```python
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

model = AutoModelForCausalLM.from_pretrained(...)  # Load the base model pretrained weights using HuggingFace Transformers
lora_config = LoraConfig(...)                      # Define the LoRA adapter architecture specifying rank, lora_alpha, target_modules, dropout, task_type
model = get_peft_model(model, lora_config)         # Inject LoRA adapters into the model, which will freeze all base model weights, and insert LoRA A and B matrices into target layers, wraps the model in PEFTModel wrapper
trainer = Trainer(...)                             # Create the HuggingFace Trainer
trainer.train()                                    # Follows the standard transformer training loop and only LoRA parameters receive gradients
```


## c. **[`trl` (2020)](https://github.com/huggingface/trl)** (Transformer Reinforcement Learning)
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px"><b>trl</b> is a HF library that provides complete training pipelines for every <b>post-pretraining stage</b> of LLM development (SFT, DPO, GRPO, PPO, RLHF and reward modeling) built on top of transformers and peft. (pre-training is outside its scope)</div></h3>

- **Problem it solves:** `transformers`'s `Trainer` is a general-purpose training loop as it has no concept of instruction masking, preference pairs, reward models, or policy optimization. `trl` adds all of these on top as ready-to-use trainers
- **What it is:** Hugging Face library that provides ready-to-use trainers for every stage of LLM training — SFT, RLHF, DPO, PPO, GRPO etc.
- **When to use:** Whenever you are doing any form of LLM fine-tuning, `SFTTrainer` alone makes training dramatically simpler than raw `Trainer`
- **Sits on top of:** `transformers` + `peft` and extends `Trainer` with LLM-specific training workflows
- **Important Links:**
    - 🔗 [Documentation](https://huggingface.co/docs/trl)
    - 🔗 [GitHub](https://github.com/huggingface/trl)
- **Trainers in `trl`:**

| Trainer | Full Name | Description |
|---|---|---|
| `SFTTrainer` | **Supervised Fine-Tuning** Trainer | Teaches the model to follow instructions by training on (instruction, response) pairs — automatically handles dataset formatting, chat templates, response-only loss masking, and packing |
| `DPOTrainer` | **Direct Preference Optimization** Trainer | Aligns model with human preferences by training on (chosen, rejected) response pairs — simpler and lower compute than PPO as no separate reward model is needed |
| `PPOTrainer` | **Proximal Policy Optimization** Trainer | Full RLHF pipeline — uses a reward model to score responses and optimizes the model using reinforcement learning — more complex than DPO but gives finer control over alignment |
| `GRPOTrainer` | **Group Relative Policy Optimization** Trainer | Teaches the model to reason step-by-step by comparing groups of sampled responses using a reward function — used in DeepSeek-R1 |
| `RewardTrainer` | **Reward Model** Trainer | Trains a reward model on human preference data — this reward model is then used by `PPOTrainer` to generate reward signals during RLHF |

- **Example TRL Pipeline (Automated LoRA Injection using`SFTTrainer`):**
```python
# HuggingFace Transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

model = AutoModelForCausalLM.from_pretrained(...)                            # Load the pretrained model, same HuggingFace loader used in the PEFT pipeline
lora_config = LoraConfig(...)                                                # Define LoRA  adapter architecture specifying rank, lora_alpha, target_modules, dropout, task_type
trainer = SFTTrainer(                                                        # Create SFTTrainer, that handles the chat template as well
                model            = model,
                args             = SFTConfig(...),
                train_dataset    = dataset,
                processing_class = tokenizer,                                # newer trl versions use processing_class instead of tokenizer
                peft_config      = lora_config,                              # SFTTrainer injects the LoRA adapters automatically
                )
trainer.train()                                                              # Train
```

## d. **`unsloth`** (2–5× Faster Fine-Tuning)
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px"><b>Unsloth</b> is an open-source library that makes LoRA and QLoRA fine-tuning <b>2–5× faster</b> and uses <b>60% less VRAM</b> than the standard transformers + peft + trl stack,  by rewriting critical GPU kernels in Triton, with zero loss in accuracy. It supports <b>SFT and alignment</b> but is not designed for pre-training or continued pre-training on raw text corpora.</div></h3>


- **Problem it solves:** The standard `transformers` + `peft` + `trl` stack,  while correct and flexible, is not maximally efficient. It uses PyTorch's default attention and backward pass implementations which have room for significant optimization. Unsloth replaces these with hand-written Triton GPU kernels that are 2–5× faster
- **What it is:** An open-source library that dramatically speeds up LoRA/QLoRA fine-tuning by rewriting attention and backpropagation kernels in Triton (low-level GPU code)
    - **2–5× faster** training than standard `transformers` + `peft`
    - **60% less VRAM** — enables fine-tuning larger models on smaller GPUs
    - Drop-in replacement for `transformers` + `peft` — same API, just faster
    - Provides pre-quantized `bnb-4bit` model versions on HF Hub
    - Supports saving to GGUF format directly for local deployment
- **When to use:** When training speed and VRAM efficiency matter — especially on free Colab T4
- **Sits on top of:** all three — `transformers`, `peft`, and `trl`
- **Limitation:** Only supports specific model architectures (Llama, Mistral, Qwen, Phi, Gemma etc.) — not every model is supported
- **Important Links:**
    - 🔗 [Documentation](https://docs.unsloth.ai)
    - 🔗 [GitHub](https://github.com/unslothai/unsloth)
- **Example Unsloth Pipeline (Optimized Training):**
```python
model, tokenizer = FastLanguageModel.from_pretrained(...)          # Load optimized model
model = FastLanguageModel.get_peft_model(...)                      # Inject LoRA adapters, internally performs PEFT LoRA injection, base model freezing and uses optimized LoRA kernels
trainer = SFTTrainer(                                              # Create trainer (still uses TRL SFTTrainer)
                model            = model,
                args             = SFTConfig(...),
                train_dataset    = dataset,
                processing_class = tokenizer,
                )
trainer.train()                                                    # Same logical training loop, but using Unsloth-optimized CUDA kernels
```

# <span style='background :lightgreen' >7. [A Deep Dive into the Working of LoRA - Low-Rank Adaptation](https://arxiv.org/abs/2106.09685)</span>

## a. LLaMA-7B Model Weights/Parameters
- A 7B model (like LLaMA-7B) has a hidden size of 4096, having 32 layers, a vocab of 32000 and MLP expansion of 11008 (SwiGLU).
- It uses RoPE (parameter-free) for positional-encoding and uses bias-free linear layers.
- LLaMA uses RMSNorm instead of LayerNorm for efficiency and stability. It uses a vector of size `hidden_dim` (e.g., 4096) that performs element-wise scaling after normalization. (scale-invariant normalization + learnable feature-wise reweighting).
- LLaMA-7B has many different weight matrices of varying shapes as shown below:
```
    - Embedding matrix:                    32000 × 4096  = 131,072,000 parameters × 1                  = 131,072,000
    - Attention Q projection:              4096  × 4096  = 16,777,216  parameters × 32 layers          = 536,870,912
    - Attention K projection:              4096  × 4096  = 16,777,216  parameters × 32 layers          = 536,870,912
    - Attention V projection:              4096  × 4096  = 16,777,216  parameters × 32 layers          = 536,870,912
    - Attention output projection:         4096  × 4096  = 16,777,216  parameters × 32 layers          = 536,870,912
    - MLP gate projection (SwiGLU):        4096  × 11008 = 45,088,768  parameters × 32 layers          = 1,442,840,576
    - MLP up projection:                   4096  × 11008 = 45,088,768  parameters × 32 layers          = 1,442,840,576
    - MLP down projection:                 11008 × 4096  = 45,088,768  parameters × 32 layers          = 1,442,840,576
    - RMSNorm (2 per layer):               (2 x 4096)    = 8192        parameters × 32 layers          = 262,144 + 4096 (final RMSNorm)
    - LM Head:                             32000 × 4096  = 131,072,000 parameters × 1 (NOT tied to EM) = 131,072,000
```
- This sums up to:
```
Embedding:          131,072,000
Attention:        2,147,483,648
MLP:              4,328,521,728
RMSNorm:                266,240
LM Head:            131,072,000
--------------------------------
TOTAL:            6,738,415,616 ≈ 6.74B
```
>- LLaMA-7B does **not** tie its LM Head to the embedding matrix, so the LM Head's 131,072,000 parameters are counted separately.
>- "7B" is just the rounded name of the model; its exact size is 6,738,415,616 ≈ 6.74B parameters (the LLaMA paper reports it as 6.7B).

## b. The Problem — Why Full Fine-Tuning is Impractical
- A 7B model in fp16 requires **14 GB** just to store the weights at inference time
- Full fine-tuning with Adam in mixed precision needs 16 bytes per parameter instead of 2, i.e., 8× more memory than inference (a model that runs in 14 GB needs 112 GB to fine-tune!) — and that is before counting activations
> 🔑 **Mixed-precision training (assumed in every memory calculation of this notebook):**
>- The forward and backward passes run on an **fp16 working copy** of the weights, so the **gradients** they produce are **fp16** too.
>- The optimizer keeps an **fp32 master copy** of every **trainable** weight, plus Adam's **momentum (m)** and **variance (v)** in **fp32**. It updates the fp32 master copy and then casts it back to fp16 for the next forward pass.
>- Gradients, master weights and Adam states are needed **only for trainable parameters**. A frozen parameter needs only its fp16 copy.
>- Per trainable parameter: 2 (fp16 weight) + 2 (fp16 gradient) + 4 (fp32 master weight) + 4 (fp32 m) + 4 (fp32 v) = **16 bytes**
>- Memory in this section is in decimal units (1 GB = 10⁹ bytes) and does not include activations.

- The complete memory breakdown for full fine-tuning a 7B model looks like this:
- **FULL FINE-TUNING — 7B Model Memory Breakdown with Adam (Adaptive Moment Estimation) Optimizer:**
```
─────────────────────────────────────────────────────────────────
Component               │ Precision │ Bytes/param │ Total Memory
────────────────────────┼───────────┼─────────────┼─────────────
1. Model Weights        │ fp16      │ 2 bytes     │  14.0 GB
2. Gradients            │ fp16      │ 2 bytes     │  14.0 GB
3. fp32 Master Weights  │ fp32      │ 4 bytes     │  28.0 GB
4. Adam Momentum (m)    │ fp32      │ 4 bytes     │  28.0 GB
5. Adam Variance (v)    │ fp32      │ 4 bytes     │  28.0 GB
─────────────────────────────────────────────────────────────────
TOTAL                                             │ 112.0 GB
─────────────────────────────────────────────────────────────────
```
>- Free tier of Google's Colab provides you a T4 with 16 GB of VRAM (~15 GB usable), which is OK for inference, but not for fine-tuning of a 7B model
>- Colab Pro+ gives you access to an A100 GPU with either 40GB or 80GB VRAM — both are still far short of the 112 GB required
>- Cloud platforms like AWS or Google Cloud, renting an 8× A100 instance costs roughly `\$29 – \$33` per hour on-demand — and that is just for the compute, before storage and data transfer fees
>- Purchasing a single A100 GPU costs approximately `\$8000 – \$15000`, depending on VRAM variant (40GB vs 80GB)


#### 💡 Understanding Why Each Component Exists
- Before understanding how LoRA solves the problem, it is important to understand WHY each of the a/m 5 memory components are needed in the first place:
    - **Model Weights (14 GB):** The actual 7B parameters are stored in fp16 that define every computation the model performs during the forward pass.
    - **Gradients (14 GB):** During the backward pass, every trainable weight needs a gradient, a number telling Adam *"update this weight by this amount"*. The gradients come out of the fp16 backward pass, so they are fp16. In full fine-tuning every weight is trainable: `7B × 2 bytes (fp16) = 14 GB` just for gradients
    - **fp32 Master Weights (28 GB):** Mixed precision training keeps a full fp32 copy of every trainable weight alongside the fp16 working copy. This is because fp16 keeps only ~3 significant decimal digits: near 1.0 the gap between two neighbouring fp16 numbers is ≈ 0.001, so a tiny update such as `1.0 + 0.000001` rounds straight back to `1.0` and the update is lost — training stalls. So Adam applies the updates to the fp32 master copy (~7 significant digits), then casts it back to fp16 for the next forward pass
    - **Adam Momentum (28 GB):** Momentum in Adam controls the *direction of the movement*. Instead of only looking at the current gradient, it keeps a running average of past gradients: `m = 0.9 × m_previous + 0.1 × current_gradient`. This smooths the learning path, reduces zig-zag movement, and helps the model reach the minimum faster and more steadily.
    - **Adam Variance (28 GB):** Variance in Adam controls the *effective step size (learning rate)* for each parameter. The Adam optimizer tracks the exponentially weighted average of squared gradients: `v = 0.999 × v_previous + 0.001 × gradient²`. If a weight’s gradients are large or noisy, updates become smaller. If a weight’s gradients are small and stable, updates can be larger.

> 💡 **Why fp32 for optimizer states?** Adam's momentum and variance are running averages accumulated over many steps. If stored in fp16, very small values underflow to zero — e.g., the variance stores squared gradients, and a gradient of 0.0001 squared is 10⁻⁸, which is below the smallest positive fp16 number (≈ 6 × 10⁻⁸) — and the small per-step changes to the running averages are lost to rounding, making training unstable. fp32 is mandatory here for numerical stability.


## c. What is Rank of a Matrix?
- The maximum number of `linearly independent` columns (or rows) of a matrix is called its rank. (A set of columns is linearly dependent if at least one of them can be made by scaling and adding the others)
- In other words: rank counts how many rows/columns cannot be derived from the other rows/columns by simple multiplication or addition
- Key rules to remember:
    - The rank of an $m×n$ matrix is always $\leq \min(m, n)$
    - A zero matrix (all elements are 0) has rank $=0$
    - Any matrix with at least one non-zero element has rank $\geq 1$
- Linear Algebra techniques to find the rank of a matrix are:
    - Row Echelon Form (REF) via Gaussian Elimination
    - Reduced Row Echelon Form (RREF) via Gauss–Jordan Elimination
    - Column Echelon Form (CEF) via Column Reduction

**Example 1:** Consider the following hypothetical $4 \times 5$ matrix (say containing weights of a neural network model). Can you identify its rank using pen and paper?

$\hspace{3 cm}W_{4,5} = \begin{bmatrix} 
       1 & 2 & 5 & 3 & 4\\ 
       3 & 3 & 9 & 6 & 9\\
       2 & 3 & 8 & 5 & 7\\
       4 & 1 & 6 & 5 & 9
\end{bmatrix}$

> 💡 **Hint:** Look carefully at the columns. Can you express any column as a combination of the other columns? Try to find the pattern focusing on columns 1 and 2 first.

<details>
<summary>✅ Click to reveal the answer</summary>
<br>

**Try expressing col3, col4, col5 using col1 and col2:**
- $col_3 = col_1 + 2 \times col_2$
- $col_4 = col_1 + col_2$
- $col_5 = 2 \times col_1 + col_2$

**Conclusion:**
- $col_3$, $col_4$, $col_5$ are **linearly dependent** on $col_1$ and $col_2$
- Only $col_1$ and $col_2$ are truly **linearly independent**
- Therefore: $\text{rank}(W) = \mathbf{2}$
</details>

In [1]:
import numpy as np
W = np.array([
    [1, 2, 5, 3, 4],
    [3, 3, 9, 6, 9],
    [2, 3, 8, 5, 7],
    [4, 1, 6, 5, 9]
])
print(W)
print(f"Maximum possible rank   : min({W.shape[0]}, {W.shape[1]}) = {min(W.shape)}")
print(f"Actual rank of W        : {np.linalg.matrix_rank(W)}") # Uses SVD (Singular Value Decomposition) internally, which is numerically more stable than the elimination methods listed above

[[1 2 5 3 4]
 [3 3 9 6 9]
 [2 3 8 5 7]
 [4 1 6 5 9]]
Maximum possible rank   : min(4, 5) = 4
Actual rank of W        : 2


## d. What is Low-Rank Decomposition?

<h3 align="center"><div class="alert alert-success" style="margin: 20px">
<b>Low-Rank Decomposition</b> is the process of approximating or exactly representing a large matrix as a product of two smaller matrices $A_{r \times d_{in}}$ and $B_{d_{out} \times r}$
</div></h3>

- We all have done factorization or prime factorization of integers:
    - $ 18 = 9 * 2$
    - $ 18 = 3 * 3 * 2$
    - $ 18 = 6 * 3$
    - $ 18 = 2 * 3 * 3$
- Similar to factorizing an integer into its prime factors, Matrix decomposition is the process of **reducing a matrix to its constituent parts** in order to make certain subsequent matrix calculations simpler. 
- **Matrix Decomposition Techniques:**
    - LU Decomposition: Factorizes a matrix into a lower triangular matrix (L) and an upper triangular matrix (U) to simplify solving linear systems.
    - LR Decomposition: A variant of LU decomposition where the matrix is expressed as a product of lower (L) and upper (R) triangular matrices.
    - QR Decomposition: Factorizes a matrix into an orthogonal matrix (Q) and an upper triangular matrix (R) to reveal independent directions in data.
    - Eigen Decomposition: Decomposes a square matrix into eigenvectors and eigenvalues to identify directions that remain unchanged under transformation.
    - Singular Value Decomposition (SVD): Decomposes any matrix into orthogonal components and singular values to capture the most important structure in the data.
- **Common Use Cases of Low-Rank Matrix Decomposition:**
    - Dimensionality Reduction: Techniques like PCA (based on SVD) reduce the number of features while keeping most of the important information (variance) in the data.
    - Data Compression: Large datasets (e.g., images) can be approximated using a low-rank version by keeping only the most important components. This significantly reduces storage while preserving key information.
    - Image Processing: Low-rank approximation helps remove noise and redundancy in images. It is commonly used in image denoising, background removal, and face recognition (Eigenfaces).
    - Fine-Tuning of LLMs (LoRA): Low-rank decomposition is used in techniques like Low-Rank Adaptation (LoRA) to efficiently fine-tune large language models. Instead of updating the entire model, only small low-rank matrices are trained, which reduces memory usage and computation cost.

>- [Applied Linear Algebra - I](https://www.youtube.com/watch?v=CLGbFiZStJU&list=PL7B2bn3G_wfCMuly7_NzKfgxon1wPUe8a&index=2) 
>- [Applied Linear Algebra - II](https://www.youtube.com/watch?v=Z2WDUUHkof8&list=PL7B2bn3G_wfCMuly7_NzKfgxon1wPUe8a&index=1) 

**Example 2:** 
- Given the following weight matrix $W_{4 \times 5}$, which has a rank of 2.
- This means $W$ can be **exactly decomposed** into two smaller matrices $A_{2 \times 5}$ and $B_{4 \times 2}$ such that $B \times A = W$, where:

$A \in R^{2 \times d_{in}} = R^{2 \times 5} \quad   d_{in}  = 5 \quad \text{(size of input vector i.e., number of columns of W)}$

$B \in R^{d_{out} \times 2} = R^{4 \times 2} \quad  d_{out} = 4 \quad \text{(size of output vector i.e., number of rows of W)}$

$W_{4 \times 5} = \begin{bmatrix} 
1 & 2 & 5 & 3 & 4\\ 
3 & 3 & 9 & 6 & 9\\
2 & 3 & 8 & 5 & 7\\
4 & 1 & 6 & 5 & 9
\end{bmatrix} = B_{4 \times 2} \times A_{2 \times 5}$

> 💡 **Hint:** From Example 1, we know that $col_1$ and $col_2$ of $W$ are the only two independent columns.  
> Use these columns to construct matrix $B$ (basis matrix). Then express each column of $W$ as a linear combination of these basis columns to construct matrix $A$, such that $B \times A$ reconstructs $W$.


<details>
<summary>✅ Click to reveal the answer</summary>
<br>

- **Step 1 — Build matrix $B$ using the two independent columns of $W$:**
$B_{4 \times 2} = \begin{bmatrix} 1 & 2\\ 3 & 3\\ 2 & 3\\ 4 & 1 \end{bmatrix}$
- **Step 2 — Build matrix $A$ using the coefficients found in Example 1:**
From Example 1, we know how each column of $W$ is built from $col_1$ and $col_2$:
- $col_1 = 1 \times col_1 + 0 \times col_2$
- $col_2 = 0 \times col_1 + 1 \times col_2$
- $col_3 = 1 \times col_1 + 2 \times col_2$
- $col_4 = 1 \times col_1 + 1 \times col_2$
- $col_5 = 2 \times col_1 + 1 \times col_2$
- These coefficients become the **columns of $A$**:
$A_{2 \times 5} = \begin{bmatrix} 1 & 0 & 1 & 1 & 2\\ 0 & 1 & 2 & 1 & 1 \end{bmatrix}$
- **Step 3 — Verify that $B \times A = W$:**
$B \times A = \begin{bmatrix} 1 & 2\\ 3 & 3\\ 2 & 3\\ 4 & 1 \end{bmatrix} \times \begin{bmatrix} 1 & 0 & 1 & 1 & 2\\ 0 & 1 & 2 & 1 & 1 \end{bmatrix} = \begin{bmatrix} 1 & 2 & 5 & 3 & 4\\ 3 & 3 & 9 & 6 & 9\\ 2 & 3 & 8 & 5 & 7\\ 4 & 1 & 6 & 5 & 9 \end{bmatrix} = W \quad$
- **Conclusion:**
- $W_{4 \times 5}$ (20 values) is **exactly reproduced** by $B_{4 \times 2} \times A_{2 \times 5}$ (8 + 10 = **18 values**)
- Similarly, **if** a $W_{31623 \times 31623}$ matrix (~1 billion or $10^9$ values) has a rank of 16 or less, it can be **exactly reproduced** by $B_{31623 \times 16} \times A_{16 \times 31623}$ (505,968 + 505,968 = **1,011,936 values**) — a saving of about **99.9%** 🔥. If its rank is higher than 16, the same two matrices can only **approximate** $W$
- This is the essence of **low-rank decomposition** — a large matrix represented by two much smaller matrices
- In LoRA, instead of updating the full $W_{m \times n}$ we learn small matrices $A$ and $B$ where the rank $r \ll \min(m, n)$
</details>

## e. Why Rank Matters for LoRA?
- In one layer of LLaMA-7B a single **Q- projection weight matrix** has (4096 × 4096) 16,777,216 weights, that are stored and full rank would mean rank = 4096
- The LoRA paper says that when you fine-tune a large model on a specific task, the *change* in weights (ΔW) that the model needs to learn has a very **low intrinsic rank**, i.e., the weight update matrix can be accurately approximated using two much smaller matrices
- In practice: effective rank of ΔW ≈ 1 to 64 out of a possible maximum rank of 4096 
- In other words: **you do not need to update all 7B parameters to teach the model a new task — the actual new information needed is much smaller**
- Think of weight updates as DIRECTIONS in space:
    - r = 1  → model can only learn changes along 1 direction. Very limited — simple tasks only
    - r = 4  → model can learn changes along 4 directions. Good for simple fine-tuning tasks
    - r = 16 → model can learn changes along 16 directions. Most commonly used value for rank
    - r = 64 → model can learn changes along 64 directions. More expressive but uses more memory. Needed for complex tasks with diverse data
    - r = 4096 → full rank — every possible direction. This is full fine-tuning — maximum expressiveness

## f. How LoRA Exploits The Low-Rank Decomposition?
```
Full Fine-Tuning (full-rank update — wasteful):           Low-rank decomposition (LoRA's approach):
───────────────────────────────────────────────────────────────────────────────────────────────────────
W_new = W_original + ΔW                                   W_new = W_original + (α/r) × B × A    (α/r is explained in Step 3 below)

Where:                                                    Where:
  W_original → NOT frozen (all entries are trained)              W_original → frozen (never updated)
  ΔW         → full size matrix (e.g., 4096 × 4096)              ΔW         → (α/r) × B × A, built from two tiny matrices
             = 16,777,216 trainable params                       A is (r × d_in) and B is (d_out × r)
  Full rank would mean, rank = 4096                              If r = 16 (r controls how many independent "directions of change" the model can learn):
                                                                     params in B and A = (4096×16) + (16×4096) = 131,072 params (128× fewer params!)
```

## g. Step by Step Working of LoRA
- **[TinyLlama-1.1B-Chat (Instruction-tuned)](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0)** is a decoder-only Transformer language model based on the LLaMA architecture. It contains 22 transformer decoder layers, a 2048 hidden size, and uses 32 attention heads. It uses Grouped-Query Attention (GQA) with 32 query heads and 4 key-value heads, along with RMSNorm for normalization, SwiGLU feed-forward networks, Rotary Positional Embeddings (RoPE), and a 2048-token context window.
```
Each Decoder Layer
├── Self-Attention
│   ├── Q  (Query projection matrix)   ← W_q    shape: 2048 × 2048   ( 32 heads   × 64 head_dim)
│   ├── K  (Key projection matrix)     ← W_k    shape: 2048 × 256    ( 4 KV heads × 64 head_dim)
│   ├── V  (Value projection matrix)   ← W_v    shape: 2048 × 256    ( 4 KV heads × 64 head_dim)
│   └── O  (Output projection matrix)  ← W_o    shape: 2048 × 2048
└── Feed-Forward Network (SwiGLU)
    ├── Gate matrix         ← W_gate  shape: 2048 × 5632
    ├── Up matrix           ← W_up    shape: 2048 × 5632
    └── Down matrix         ← W_down  shape: 5632 × 2048
```
- **Note:**
>- In the original transformer architecture, each attention head has its own Q, K and V.
>- Grouped-Query Attention (GQA) is a variant of multi-head attention where multiple query heads share a smaller number of key and value heads to reduce computation and memory while maintaining performance.
>- In TinyLlama, Q has 32 heads but K and V only have 4 heads. This means W_q and W_o are 2048 × 2048 while W_k and W_v are only 2048 × 256.
>- To keep things simple in the steps below, we will only inject LoRA into W_q and W_o (both 2048 × 2048) and avoid W_k and W_v.

- **Step 1 — Freeze the entire TinyLlama-1.1B base model:**
    - All original pretrained weights across all **22 decoder layers** are locked and will never be updated.
    - This preserves all the general language knowledge learned during TinyLlama's pretraining on **3 trillion tokens**
    - TinyLlama's original weight matrices (e.g. `q_proj`, `o_proj` of shape `2048 × 2048`) become pure **read-only** during fine-tuning<br><br>
- **Step 2 — Inject LoRA adapter matrices A and B into selected layers:**
    - For each of the **22 decoder layers**, two pairs of small matrices are inserted alongside the selected projection matrices (typically `q_proj` and `o_proj`):
        - **Matrix A** → shape `(16 x 2048)` — initialized with random Gaussian values
        - **Matrix B** → shape `(2048 x 16)` — initialized with **zeros**
    - Starting $B$ at zero ensures $\Delta W = B \times A = 0$ at the start — meaning fine-tuning begins from **exactly the same behavior** as the original TinyLlama model with no sudden disruption
    - Total trainable parameters: 22 layers $\times$ 2 projections $\times (2048 \times 16 + 16 \times 2048) = 2883584 \approx$ **~2.9M** (out of 1.1B, only ~0.26% of the model) 🔥 <br><br>
- **Step 3 — Train ONLY matrices A and B:**
    - During each forward pass through TinyLlama, the output of each adapted layer becomes:<br>
$\text{output} = \underbrace{W_{\text{original}} \times x}_{\text{frozen path}} + \underbrace{(B \times A) \times x \times \frac{\alpha}{r}}_{\text{LoRA adapter path}}$
    - Only $A$ and $B$ receive gradients — the frozen `q_proj` and `o_proj` weight matrices of TinyLlama **never change**
    - $\alpha$ (`lora_alpha`) is a scaling factor controlling how strongly the adapter influences TinyLlama's output — typically set to `2 x r`
    - $r$ (`lora_rank`) controls the rank of the decomposition — typically set to `8` or `16` <br><br>
- **Step 4 — Merge adapters for inference (optional):**
    - After fine-tuning is complete, the adapter can be merged back into TinyLlama's original weights:
$W_{\text{merged}} = W_{\text{original}} + \frac{\alpha}{r} . B \times A$
    - The result is a single standard TinyLlama-style weight matrix with **zero inference overhead** — no adapter logic remaining at runtime
    - Alternatively, keep $A$ and $B$ **separate and swappable** — allowing you to maintain one frozen TinyLlama base and **hot-swap multiple LoRA adapters** for different tasks (e.g. one for coding, one for medical QA, one for Urdu language) without reloading the entire model

## h. Key LoRA Hyperparameters
- **`target_modules` — Which Layers to Apply LoRA To**
    - Typically applied to attention projection layers:
        `q_proj`, `k_proj`, `v_proj`, `o_proj`
    - Can also include MLP layers: `gate_proj`, `up_proj`, `down_proj`
    - More modules → better results but more memory
- **`r` — Rank:**
    - Controls the size of adapter matrices A and B
    - Smaller r → fewer params → faster, less memory, less expressive
    - Larger r → more params → slower, more memory, more expressive
    - Typical values: `4, 8, 16, 32`
- **`lora_alpha` — Scaling Factor**
    - Controls how much the adapter influences the output via `α/r`
    - Common practice: set `lora_alpha = 2 × r` (e.g., if r=16 → alpha=32)
- **`lora_dropout`**
    - It's a regularization technique that helps prevent the adapters from overfitting to the (usually small) fine-tuning dataset.
    - Typical values are `0.05`–`0.1`. With `0.1`, during training 10% of the values of the input `x` that flows into the LoRA branch (`B × A × x`) are randomly set to zero at every step (a different random 10% each time). The frozen path `W × x` is not affected, and dropout is switched off at inference.
    - Unsloth supports any value, but its fast kernels are optimized for `0`, so Unsloth notebooks usually set `lora_dropout = 0`

## i. LoRA Architecture Diagram
```
  Input x
     │
     ▼
┌────────────────────────────────────────────┐
│          Pretrained Weight W               │
│         (FROZEN — never updated)           │
│              d × d matrix                  │
└────────────────────────────────────────────┘
     │                    │
     │              ┌─────▼──────┐
     │              │  Matrix A  │  ← trained │ r × d │ initialized: random
     │              └─────┬──────┘
     │                    │  (low-rank bottleneck)
     │              ┌─────▼──────┐
     │              │  Matrix B  │  ← trained │ d × r │ initialized: zeros
     │              └─────┬──────┘
     │                    │  × (α / r)   scaling
     ▼                    ▼
  W · x        +        B · A · x
     │                     │
     └──────────┬──────────┘
                ▼
            Output h
   (pretrained + adapter contribution)
```

## j. How LoRA Solves the 112 GB Problem

<h3 align="center"><div class="alert alert-success" style="margin: 20px">LoRA does not shrink the model itself—it reduces what needs to be trained and stored during training.</div></h3>


- LoRA attacks 4 out of 5 memory components — the only component it cannot reduce is the model weights themselves (since the base model must still be loaded in fp16 for the forward pass)
    - **Base weights (14 GB → 14 GB — unchanged):** LoRA freezes the base model but still needs to load all 7B weights in fp16 for the forward pass
    - **Gradients (fp16):** Computed and stored only for the small adapter matrices — frozen weights receive no weight gradients
    - **Master weights (fp32):** Kept only for the small adapter matrices — frozen weights are never updated, so they need no fp32 copy
    - **Adam momentum (fp32):** Kept only for the small adapter matrices
    - **Adam variance (fp32):** Kept only for the small adapter matrices
- The adapters also add a little memory of their own: every adapter parameter is trainable, so it needs 2 (fp16 weight) + 2 (fp16 gradient) + 4 (fp32 master weight) + 4 (fp32 m) + 4 (fp32 v) = 16 bytes
- **Example:** LoRA with `r = 16` on all 7 projection matrices (`q, k, v, o, gate, up, down`) in all 32 layers of LLaMA-7B:
```
Adapter params per layer:
  q, k, v, o     (4096 × 4096 each)  : 4 × 16 × (4096 + 4096)  =   524,288
  gate, up, down (4096 ↔ 11008 each) : 3 × 16 × (4096 + 11008) =   724,992
                                                                 ─────────
                                                                 1,249,280 × 32 layers = 39,976,960 params (≈ 40M, ≈ 0.6% of 6.74B)

Adapter memory = 39,976,960 × 16 bytes                  ≈  0.64 GB
Total memory   = 14 GB (frozen fp16 base) + 0.64 GB     ≈ 14.6  GB   (activations not included)
```

⚠️ **LoRA alone brings 112 GB down to ~14.6 GB (≈ 87% reduction)**

>- A massive improvement, but still too tight for a free Colab T4 GPU (16 GB VRAM, ~15 GB usable) once activations are added.
>- The remaining bottleneck is the **14 GB of fp16 model weights** which LoRA cannot reduce — that is exactly the problem that **QLoRA solves** by loading the base model in NF4 4-bit (0.5 bytes/param) instead of fp16 (2 bytes/param): 7B × 0.5 bytes = 3.5 GB, plus ~0.64 GB for the adapters, bringing the total down to **~4 GB**.

# <span style='background :lightgreen' >8. Example Questions to Check your Understanding about LoRA</span>


# Sample Question 01
- You are fine-tuning a transformer model that has a single attention layer with the following weight matrices:

```
Query projection  : W_Q  shape = (2048 × 2048)
Key projection    : W_K  shape = (2048 × 2048)
Value projection  : W_V  shape = (2048 × 2048)
Output projection : W_O  shape = (2048 × 2048)
```
- You decide to apply LoRA with rank **r = 8** to all four matrices.
- Training uses **mixed precision** with the **Adam optimizer**, exactly as described in Section 6(b):
    - **Weights** are stored in **fp16** (2 bytes/param). This covers the four base weight matrices (trainable in full fine-tuning, frozen in LoRA) and the LoRA adapter matrices A and B.
    - **Gradients** are produced by the fp16 backward pass, so they are also **fp16** (2 bytes/param). They exist **only for trainable parameters**.
    - For every **trainable** parameter only, the optimizer also keeps an **fp32 master copy** of the weight, an **fp32 Adam momentum (m)** and an **fp32 Adam variance (v)** (4 bytes/param each).
    - **Frozen** parameters need only their fp16 copy — no gradient, no master copy and no Adam states.
- Assume **no quantization** (plain LoRA, not QLoRA) and **ignore activation memory**.
- Give all memory values in **MiB**, where 1 MiB = 2²⁰ = 1,048,576 bytes.
---

- **(a)** How many parameters does ONE weight matrix (e.g., W_Q) have?
- **(b)** How many total parameters are there in all four matrices combined?
- **(c)** What is the shape of LoRA matrices and total trainable parameters across all four matrices (Q, K, V, O)?
- **(d)** What percentage of the original 4-matrix parameters are actually trained via LoRA?
- **(e)** How much memory do the 4 frozen base weight matrices consume in fp16?
- **(f)** How much memory do ALL LoRA adapter weights (A and B of all four matrices) consume in fp16?
- **(g)** Calculate and fill in the following comparison table. In full fine-tuning, all 16,777,216 base weights are trainable; in LoRA, only the adapter weights are trainable:

```
Component                    │ Full Fine-Tuning │ LoRA (r=8) │ Saving
─────────────────────────────┼──────────────────┼────────────┼────────
Base weights (fp16)          │                  │            │
LoRA adapter weights (fp16)  │                  │            │
Gradients (fp16)             │                  │            │
fp32 master weights          │                  │            │
Adam momentum (fp32)         │                  │            │
Adam variance (fp32)         │                  │            │
─────────────────────────────┼──────────────────┼────────────┼────────
TOTAL                        │                  │            │
```

- **(h)** At initialization, Matrix A is initialized with random Gaussian values and Matrix B is initialized with all zeros. What is the value of the LoRA update `ΔW = B × A` at initialization? Why is this initialization strategy important?
- **(i)** After LoRA training is complete, you have two options for inference:
    - **Option 1:** Keep base model + load LoRA adapters separately
    - **Option 2:** Merge adapters into base weights: `W_new = W_base + (α/r) × B × A`
    - For each option state:
        - How much memory is needed? (base weights + adapter weights if any)
        - Is there any computational overhead at inference time?
        - When would you prefer each option?

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
Given:
  Matrix shape       = 2048 × 2048
  Number of matrices = 4  (W_Q, W_K, W_V, W_O)
  LoRA rank          = r = 8
  fp16               = 2 bytes/param → weights (base + adapters) and gradients
  fp32               = 4 bytes/param → master weights, Adam momentum, Adam variance
  Trainable params   → fp16 weight + fp16 gradient + fp32 master + fp32 m + fp32 v = 16 bytes/param
  Frozen params      → fp16 weight only = 2 bytes/param
  1 MiB              = 2^20 = 1,048,576 bytes


(a) Parameters in ONE weight matrix:
    W_Q = 2048 × 2048 = 4,194,304 params

(b) Total parameters in all four matrices:
    4 × 4,194,304 = 16,777,216 params

(c) LoRA matrix shapes and trainable parameter count:
    For each weight matrix of shape (2048 × 2048):
      A shape = (r    × d_in)  = (8    × 2048) =  16,384 params
      B shape = (d_out × r   ) = (2048 × 8   ) =  16,384 params
      LoRA params per matrix   =  16,384 + 16,384 = 32,768 params

    Across all four matrices (W_Q, W_K, W_V, W_O):
      Total LoRA trainable params = 4 × 32,768 = 131,072 params

(d) Percentage of original parameters trained via LoRA:
    = (131,072 / 16,777,216) × 100 = 0.78125%
                        → Using LoRA, we train around 0.78% of the original parameters!

(e) Memory for 4 frozen base weight matrices in fp16:
    = 16,777,216 × 2 bytes = 33,554,432 bytes = 33,554,432 / 2^20 = 32 MiB

(f) Memory for ALL LoRA adapter weights in fp16:
    = 131,072 × 2 bytes = 262,144 bytes = 262,144 / 2^20 = 0.25 MiB


(g) Comparison Table calculations:

  ── Full Fine-Tuning (all 16,777,216 params trainable) ──────────────────────
  Base weights (fp16)  = 16,777,216 × 2 = 33,554,432 bytes = 32.0 MiB
  Gradients (fp16)     = 16,777,216 × 2 = 33,554,432 bytes = 32.0 MiB
  fp32 master weights  = 16,777,216 × 4 = 67,108,864 bytes = 64.0 MiB
  Adam momentum (fp32) = 16,777,216 × 4 = 67,108,864 bytes = 64.0 MiB
  Adam variance (fp32) = 16,777,216 × 4 = 67,108,864 bytes = 64.0 MiB
  TOTAL = 32.0 + 32.0 + 64.0 + 64.0 + 64.0 = 256.0 MiB
  (check: 16,777,216 params × 16 bytes = 268,435,456 bytes = 256 MiB)

  ── LoRA r=8 (base frozen, only the 131,072 adapter params trainable) ───────
  Frozen base weights (fp16)  = 16,777,216 × 2 = 33,554,432 bytes = 32.0  MiB
  LoRA adapter weights (fp16) =    131,072 × 2 =    262,144 bytes =  0.25 MiB
  Gradients (fp16)            =    131,072 × 2 =    262,144 bytes =  0.25 MiB  ← adapters only
  fp32 master weights         =    131,072 × 4 =    524,288 bytes =  0.5  MiB  ← adapters only
  Adam momentum (fp32)        =    131,072 × 4 =    524,288 bytes =  0.5  MiB  ← adapters only
  Adam variance (fp32)        =    131,072 × 4 =    524,288 bytes =  0.5  MiB  ← adapters only
  TOTAL = 32.0 + 0.25 + 0.25 + 0.5 + 0.5 + 0.5 = 34.0 MiB
  (check: 16,777,216 × 2 + 131,072 × 16 = 35,651,584 bytes = 34 MiB)

  ── Savings ─────────────────────────────────────────────────────────────────
  Base weights     : 32.0  - 32.0  =   0     MiB  (0.0%  — always required)
  LoRA adapters    :  0    - 0.25  =  -0.25  MiB  (extra memory that LoRA adds)
  Gradients        : 32.0  - 0.25  =  31.75  MiB  (99.2% reduction)
  Master weights   : 64.0  - 0.5   =  63.5   MiB  (99.2% reduction)
  Adam momentum    : 64.0  - 0.5   =  63.5   MiB  (99.2% reduction)
  Adam variance    : 64.0  - 0.5   =  63.5   MiB  (99.2% reduction)
  TOTAL saving     : 256.0 - 34.0  = 222.0   MiB  (86.7% reduction)

┌─────────────────────────────┬──────────────────┬────────────┬────────────────────────┐
│ Component                   │ Full Fine-Tuning │ LoRA (r=8) │ Saving                 │
├─────────────────────────────┼──────────────────┼────────────┼────────────────────────┤
│ Base weights (fp16)         │ 32.0  MiB        │ 32.0  MiB  │ 0 MiB (always needed)  │
│ LoRA adapter weights (fp16) │ —                │  0.25 MiB  │ -0.25 MiB (extra)      │
│ Gradients (fp16)            │ 32.0  MiB        │  0.25 MiB  │ 31.75 MiB (99.2%)      │
│ fp32 master weights         │ 64.0  MiB        │  0.5  MiB  │ 63.5  MiB (99.2%)      │
│ Adam momentum (fp32)        │ 64.0  MiB        │  0.5  MiB  │ 63.5  MiB (99.2%)      │
│ Adam variance (fp32)        │ 64.0  MiB        │  0.5  MiB  │ 63.5  MiB (99.2%)      │
├─────────────────────────────┼──────────────────┼────────────┼────────────────────────┤
│ TOTAL                       │ 256.0 MiB        │ 34.0  MiB  │ 222.0 MiB (86.7%)      │
└─────────────────────────────┴──────────────────┴────────────┴────────────────────────┘


(h) LoRA update at initialization:
    A ← random Gaussian (small non-zero values)
    B ← all zeros

    ΔW = B × A = 0 × A = 0  (zero matrix, all entries = 0)

    Why this initialization strategy is important:
    - At the start of training, ΔW = 0 means the adapted model
      is IDENTICAL to the original pretrained model
    - This ensures training begins from a stable, known starting
      point — the pretrained checkpoint — without any random
      disruption to the model's existing knowledge
    - If B were also initialized randomly, ΔW would be a random
      matrix that would immediately corrupt the pretrained
      representations before any useful learning has occurred
    - The zero initialization preserves the pretrained model's
      capabilities at step 0 and lets the adapter learn only
      the task-specific delta from that clean starting point

(i) Inference Options (inference needs only fp16 weights — no gradients,
    no master weights and no Adam states):

  ┌──────────────────────┬────────────────────────────────────────────────────┐
  │ Option 1             │ Keep base model + LoRA adapters separately         │
  ├──────────────────────┼────────────────────────────────────────────────────┤
  │ Memory needed        │ Base weights : 32.0  MiB (fp16)                    │
  │                      │ LoRA adapter :  0.25 MiB (fp16)                    │
  │                      │ TOTAL        : 32.25 MiB                           │
  ├──────────────────────┼────────────────────────────────────────────────────┤
  │ Computational        │ YES — at every forward pass each adapted matrix    │
  │ overhead             │ computes:                                          │
  │                      │   output = W_base·x + (α/r) × B·(A·x)              │
  │                      │ i.e., two extra small matrix multiplications       │
  │                      │ (A·x, then B·(A·x)) per adapted matrix             │
  ├──────────────────────┼────────────────────────────────────────────────────┤
  │ When to prefer       │ Serving MULTIPLE fine-tuned tasks from the SAME    │
  │                      │    base model — swap only the 0.25 MiB adapter     │
  │                      │    between tasks without reloading the base model  │
  │                      │ Storage-constrained deployments with many tasks    │
  │                      │ During active experimentation and iteration        │
  └──────────────────────┴────────────────────────────────────────────────────┘

  ┌──────────────────────┬────────────────────────────────────────────────────┐
  │ Option 2             │ Merge adapters: W_new = W_base + (α/r) × B × A     │
  ├──────────────────────┼────────────────────────────────────────────────────┤
  │ Memory needed        │ Merged weights : 32.0 MiB (fp16)                   │
  │                      │ LoRA adapter   :  0   MiB (discarded after merge)  │
  │                      │ TOTAL          : 32.0 MiB                          │
  ├──────────────────────┼────────────────────────────────────────────────────┤
  │ Computational        │ NO — merging is a ONE-TIME operation before        │
  │ overhead             │ deployment. W_new is just a regular weight matrix. │
  │                      │ Inference is identical to a standard model         │
  ├──────────────────────┼────────────────────────────────────────────────────┤
  │ When to prefer       │ Deploying a SINGLE fine-tuned model to production  │
  │                      │    where inference latency matters                 │
  │                      │ When you want zero overhead and the simplest       │
  │                      │    possible inference pipeline                     │
  │                      │ When the base model is not shared across tasks     │
  └──────────────────────┴────────────────────────────────────────────────────┘

  Key insight: merging costs 0 extra memory (W_new is the same
  shape as W_base) and eliminates all inference overhead —
  the only trade-off is losing the ability to swap adapters
  without re-merging.
```

</details>

<br><br><br><br><br><br><br><br>
# Sample Question 02
- Consider a model with a single weight matrix W that maps a 4096-dim input to an 11008-dim output (d_in = 4096, d_out = 11008), just like the MLP up-projection of LLaMA-7B in Section 6(a). Following the LoRA convention of this notebook, W has shape **d_out × d_in = 11008 × 4096** (Section 6(a) lists the same layer as 4096 × 11008, i.e., in input × output order)
- Training uses **mixed precision**: model weights and gradients are stored in **fp16** (2 bytes/param), and the Adam optimizer keeps an **fp32 master copy**, **fp32 momentum** and **fp32 variance** (4 bytes/param each) for every **trainable** parameter. Gradients, master weights and Adam states exist only for trainable parameters; frozen weights need only their fp16 copy. Ignore activation memory.
- How much memory (in MiB, where 1 MiB = 2²⁰ bytes) do you need to hold the model weights for inference?
- Training uses the Adam optimizer. How much GPU memory (in MiB) do you need:
    - When doing full fine-tuning?
    - When doing PEFT using LoRA strategy with a rank of 8?
> Show your working to get credit


<details>
<summary>💡 <b>Click to reveal answer</b></summary>

- **Memory required for Inference:**
```
Total Parameters = 11008 × 4096 = 45,088,768 = 45088768/(10^6) = 45.088768 Million parameters
Memory required for inference = 45,088,768 × 2 bytes = 90,177,536 bytes = 90,177,536/(2^20) = 86 MiB
```
- **Memory required for Full Fine-Tuning with Adam optimizer (all 45,088,768 params trainable):**
```
1. Model Weights (fp16)   = 45,088,768 × 2 bytes  =  90,177,536 bytes =  90,177,536/(2^20) =  86.0 MiB
2. Gradients (fp16)       = 45,088,768 × 2 bytes  =  90,177,536 bytes =  90,177,536/(2^20) =  86.0 MiB
3. Master Weights (fp32)  = 45,088,768 × 4 bytes  = 180,355,072 bytes = 180,355,072/(2^20) = 172.0 MiB
4. Adam Momentum (fp32)   = 45,088,768 × 4 bytes  = 180,355,072 bytes = 180,355,072/(2^20) = 172.0 MiB
5. Adam Variance (fp32)   = 45,088,768 × 4 bytes  = 180,355,072 bytes = 180,355,072/(2^20) = 172.0 MiB
──────────────────────────────────────────────────────────────────────────────────────────────────────
TOTAL = 86 + 86 + 172 + 172 + 172                                                            = 688.0 MiB
(check: 45,088,768 params × 16 bytes = 721,420,288 bytes = 688 MiB)
```

- **Memory required using LoRA with rank of 8 with Adam optimizer (base frozen, only adapters trainable):**
```
LoRA learns the weight update ΔW as the product of two low-rank matrices (ΔW = B × A):
  Matrix A : r × d_in  = 8 × 4096    =  32,768 params
  Matrix B : d_out × r = 11008 × 8   =  88,064 params
  ─────────────────────────────────────────────────
  Total LoRA (trainable) params      = 120,832 params  (120,832 / 45,088,768 ≈ 0.27% of the weights)
  Shape check: B × A = (11008 × 8) × (8 × 4096) = 11008 × 4096 = same shape as W ✓


1. Frozen Model Weights (fp16) = 45,088,768 × 2 bytes = 90,177,536 bytes = 86.0        MiB
2. LoRA Adapters (fp16)        =    120,832 × 2 bytes =    241,664 bytes =  0.23046875 MiB
3. Gradients (fp16)            =    120,832 × 2 bytes =    241,664 bytes =  0.23046875 MiB  ← adapters only
4. Master Weights (fp32)       =    120,832 × 4 bytes =    483,328 bytes =  0.4609375  MiB  ← adapters only
5. Adam Momentum (fp32)        =    120,832 × 4 bytes =    483,328 bytes =  0.4609375  MiB  ← adapters only
6. Adam Variance (fp32)        =    120,832 × 4 bytes =    483,328 bytes =  0.4609375  MiB  ← adapters only
──────────────────────────────────────────────────────────────────────────────────────────────
TOTAL = 86 + 0.23046875 + 0.23046875 + 0.4609375 + 0.4609375 + 0.4609375   = 87.84375 MiB ≈ 87.84 MiB
(check: 45,088,768 × 2 + 120,832 × 16 = 92,110,848 bytes = 87.84375 MiB)

LoRA vs full fine-tuning: 688 - 87.84375 = 600.15625 MiB saved (≈ 87.2% less memory)
```



</details>

<br><br><br><br><br><br><br><br><br><br><br><br>

# Sample Question 03
- A feed-forward weight matrix W has shape **4096 × 11008** (d_out × d_in, i.e., it maps an 11008-dim input to a 4096-dim output, like the MLP down-projection of LLaMA-7B in Section 6(a))
- It is fine-tuned with LoRA as W + B × A (the α/r scaling factor does not change any of the counts below, so ignore it)
- Answer the following questions:
    - **(a)** How many weights does full fine-tuning update in this matrix?
    - **(b)** With rank r = 16, give the shapes of B and A, the number of trainable parameters, and that number as a percentage of your answer to part (a)
    - **(c)** What is the smallest whole-number rank r at which LoRA would train more parameters than full fine-tuning of this matrix?
> Show your working to get credit


<details>
<summary>💡 <b>Click to reveal answer</b></summary>

- **(a) Weights updated by full fine-tuning:**
```
Full fine-tuning updates every entry of W:
  4096 × 11008 = 45,088,768 weights
```
- **(b) LoRA with r = 16:**
```
B × A must have the same shape as W (4096 × 11008):
  B : d_out × r = 4096 × 16  =  65,536 params
  A : r × d_in  = 16 × 11008 = 176,128 params
  ─────────────────────────────────────────────
  Trainable params           = 241,664 params   (= 16 × (4096 + 11008) = 16 × 15,104)

Percentage of full fine-tuning = 241,664 / 45,088,768 × 100 ≈ 0.536%
```
- **(c) Break-even rank:**
```
LoRA trains    r × (d_out + d_in) = r × (4096 + 11008) = r × 15,104 params
Full FT trains d_out × d_in       = 4096 × 11008       = 45,088,768 params

LoRA trains more when:  r × 15,104 > 45,088,768
                        r > 45,088,768 / 15,104 ≈ 2985.22

  r = 2985 → 2985 × 15,104 = 45,085,440 params  (still fewer than 45,088,768)
  r = 2986 → 2986 × 15,104 = 45,100,544 params  (more than 45,088,768)

→ From r = 2986 upward, LoRA trains MORE parameters than full fine-tuning
```

> 💡 **Key insight:**
>- The largest possible rank of this matrix is min(4096, 11008) = 4096. At r = 4096, LoRA would train 4096 × 15,104 = 61,865,984 parameters, i.e., about 1.37× as many as full fine-tuning.
>- In general, LoRA trains fewer parameters only while r < (d_out × d_in) / (d_out + d_in). For a square d × d matrix this break-even rank is d / 2.
>- That is why LoRA is used with very small ranks (r = 4–64): its savings come entirely from r ≪ min(d_out, d_in).

</details>

<br><br><br><br><br><br><br><br><br><br><br><br>